# Benchmark EfficientNetB0: chuẩn bị dữ liệu và harness

Phạm vi hiện tại là Milestone 1. Notebook sử dụng Imagenette 320px làm nguồn dữ liệu, tạo manifest cố định cho 500 ảnh validation và 100 ảnh calibration, kiểm tra tiền xử lý trên ảnh thật, và định nghĩa các tiện ích đo lường. Notebook không tải pretrained weights, không chạy baseline, không chuyển đổi ONNX/TFLite và không lượng hóa INT8.


In [1]:
from __future__ import annotations

import json
import os
import platform
import random
import sys
import time
from dataclasses import asdict, dataclass
from functools import lru_cache
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import psutil
import tensorflow as tf
import onnx  # noqa: F401
import onnxruntime as ort
import tf2onnx  # noqa: F401
import matplotlib  # noqa: F401
import tqdm  # noqa: F401
import sklearn  # noqa: F401
import ipykernel  # noqa: F401
import jupyterlab  # noqa: F401
from PIL import Image, ImageOps
from tensorflow.keras.applications.efficientnet import preprocess_input as efficientnet_preprocess_input

try:
    import pynvml as nvml
except Exception:
    nvml = None


def detect_project_root(start: Path | None = None) -> Path:
    candidate = (start or Path.cwd()).resolve()
    for item in (candidate, *candidate.parents):
        if (item / "TASK.md").exists() and (item / "PLAN.md").exists() and (item / "CONTEXT.md").exists():
            return item
    return candidate


PROJECT_ROOT = detect_project_root()
DATA_ROOT = PROJECT_ROOT / "data"
IMAGENETTE_ROOT = DATA_ROOT / "raw" / "imagenette2-320"
RAW_DATA_DIR = IMAGENETTE_ROOT
METADATA_DIR = DATA_ROOT / "metadata"
MANIFEST_DIR = DATA_ROOT / "manifests"
LABEL_DIR = DATA_ROOT / "labels"
RESULTS_DIR = PROJECT_ROOT / "results"
FIGURES_DIR = PROJECT_ROOT / "figures"
LOGS_DIR = PROJECT_ROOT / "logs"

SEED = 42
BATCH_SIZE = 1
THREAD_COUNT = 1
WARMUP_RUNS = 20
BENCHMARK_SIZE = 500
CALIBRATION_SIZE = 100
INPUT_SHAPE = (224, 224, 3)
EXPECTED_KERNEL_NAME = "slot17"
EXPECTED_KERNEL_DISPLAY = "Python (slot17)"

RESULT_COLUMNS = [
    "run_id",
    "model_id",
    "runtime",
    "precision",
    "device",
    "provider",
    "batch_size",
    "thread_count",
    "num_images",
    "warmup_runs",
    "model_size_mb",
    "load_time_s",
    "total_model_only_time_s",
    "total_end_to_end_time_s",
    "mean_latency_ms",
    "median_latency_ms",
    "p95_latency_ms",
    "fps_model_only",
    "fps_end_to_end",
    "top1_accuracy",
    "top5_accuracy",
    "top1_agreement",
    "max_abs_output_diff",
    "mean_abs_output_diff",
    "ram_avg_mb",
    "ram_peak_mb",
    "cpu_process_avg_pct",
    "cpu_process_peak_pct",
    "cpu_system_avg_pct",
    "cpu_system_peak_pct",
    "gpu_avg_pct",
    "gpu_peak_pct",
    "vram_avg_mb",
    "vram_peak_mb",
    "speedup_vs_baseline",
    "size_reduction_pct",
    "accuracy_delta",
    "notes",
]

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Python:", sys.version.split()[0])
print("Interpreter:", sys.executable)
print("Kernel expectation:", EXPECTED_KERNEL_NAME, EXPECTED_KERNEL_DISPLAY)
print("Benchmark config:", {"seed": SEED, "batch_size": BATCH_SIZE, "thread_count": THREAD_COUNT, "warmup_runs": WARMUP_RUNS, "benchmark_size": BENCHMARK_SIZE, "calibration_size": CALIBRATION_SIZE})


PROJECT_ROOT:

D:\DAT301m\slot17

Python:

3.11.9

Interpreter:

...

Kernel expectation:

slot17

Python (slot17)

Benchmark config:

{'seed': 42, 'batch_size': 1, 'thread_count': 1, 'warmup_runs': 20, 'benchmark_size': 500, 'calibration_size': 100}

In [2]:
def _version(package_name: str) -> str | None:
    try:
        from importlib import metadata
        return metadata.version(package_name)
    except Exception:
        return None


def collect_environment_validation() -> dict[str, Any]:
    tf_devices = [device.name for device in tf.config.list_physical_devices()]
    cpu_devices = [device.name for device in tf.config.list_physical_devices("CPU")]
    gpu_devices = [device.name for device in tf.config.list_physical_devices("GPU")]
    build_info = tf.sysconfig.get_build_info()
    providers = ort.get_available_providers()
    packages = {
        "tensorflow": _version("tensorflow"),
        "tensorflow-intel": _version("tensorflow-intel"),
        "tf2onnx": _version("tf2onnx"),
        "onnx": _version("onnx"),
        "onnxruntime": _version("onnxruntime"),
        "numpy": _version("numpy"),
        "protobuf": _version("protobuf"),
        "pandas": _version("pandas"),
        "Pillow": _version("Pillow"),
        "psutil": _version("psutil"),
        "matplotlib": _version("matplotlib"),
        "tqdm": _version("tqdm"),
        "scikit-learn": _version("scikit-learn"),
        "ipykernel": _version("ipykernel"),
        "jupyterlab": _version("jupyterlab"),
        "nvidia-ml-py": _version("nvidia-ml-py"),
    }
    return {
        "project_root": str(PROJECT_ROOT),
        "operating_system": platform.system(),
        "os_release": platform.release(),
        "os_version": platform.version(),
        "architecture": platform.architecture()[0],
        "python_version": sys.version.split()[0],
        "python_executable": sys.executable,
        "virtual_environment": os.environ.get("VIRTUAL_ENV"),
        "cpu_name": platform.processor(),
        "cpu_count_logical": psutil.cpu_count(logical=True),
        "cpu_count_physical": psutil.cpu_count(logical=False),
        "total_ram_gb": round(psutil.virtual_memory().total / (1024 ** 3), 2),
        "tensorflow_devices": tf_devices,
        "tensorflow_cpu_devices": cpu_devices,
        "tensorflow_gpu_devices": gpu_devices,
        "tensorflow_cuda_build": bool(build_info.get("is_cuda_build", False)),
        "tensorflow_build_info": build_info,
        "onnxruntime_providers": providers,
        "packages": packages,
    }


environment_validation = collect_environment_validation()
print(json.dumps(environment_validation, indent=2, ensure_ascii=False, default=str))


{
  "project_root": "D:\\DAT301m\\slot17",
  "operating_system": "Windows",
  "os_release": "10",
  "os_version": "10.0.22631",
  "architecture": "64bit",
  "python_version": "3.11.9",
  "python_executable": "D:\\DAT301m\\slot17\\.venv-slot17\\Scripts\\python.exe",
  "virtual_environment": null,
  "cpu_name": "Intel64 Family 6 Model 154 Stepping 3, GenuineIntel",
  "cpu_count_logical": 16,
  "cpu_count_physical": 12,
  "total_ram_gb": 23.71,
  "tensorflow_devices": [
    "/physical_device:CPU:0"
  ],
  "tensorflow_cpu_devices": [
    "/physical_device:CPU:0"
  ],
  "tensorflow_gpu_devices": [],
  "tensorflow_cuda_build": false,
  "tensorflow_build_info": {
    "is_cuda_build": false,
    "is_rocm_build": false,
    "is_tensorrt_build": false,
    "msvcp_dll_names": "msvcp140.dll,msvcp140_1.dll"
  },
  "onnxruntime_providers": [
    "AzureExecutionProvider",
    "CPUExecutionProvider"
  ],
  "packages": {
    "tensorflow": "2.15.1",
    "tensorflow-intel": "2.15.1",
    "tf2onnx": "1.16

In [3]:
DATASET_PATHS = {
    "raw_images": IMAGENETTE_ROOT,
    "validation_split": IMAGENETTE_ROOT / "val",
    "calibration_split": IMAGENETTE_ROOT / "train",
    "sample_manifest": MANIFEST_DIR / "sample_500.csv",
    "calibration_manifest": MANIFEST_DIR / "calibration_100.csv",
    "imagenet_class_index": LABEL_DIR / "imagenet_class_index.json",
}

def describe_path(path: Path) -> dict[str, Any]:
    return {"path": str(path), "exists": path.exists(), "is_file": path.is_file(), "is_dir": path.is_dir()}

def validate_dataset_inputs() -> dict[str, Any]:
    required_inputs = {name: DATASET_PATHS[name] for name in ("raw_images", "validation_split", "calibration_split", "imagenet_class_index")}
    manifests = {name: DATASET_PATHS[name] for name in ("sample_manifest", "calibration_manifest")}
    missing_inputs = [name for name, path in required_inputs.items() if not path.exists()]
    return {
        "ready": not missing_inputs, "status": "READY" if not missing_inputs else "BLOCKED_ON_DATASET",
        "missing_inputs": missing_inputs,
        "required_inputs": {name: describe_path(path) for name, path in required_inputs.items()},
        "manifest_targets": {name: describe_path(path) for name, path in manifests.items()},
        "manifest_files_present": {name: path.exists() for name, path in manifests.items()},
        "expected_benchmark_size": BENCHMARK_SIZE, "expected_calibration_size": CALIBRATION_SIZE,
        "seed": SEED, "no_overlap_required": True, "label_source": "Imagenette synset folder + ImageNet class index JSON",
    }

dataset_validation = validate_dataset_inputs()
print(json.dumps(dataset_validation, indent=2, ensure_ascii=False))


{
  "ready": true,
  "status": "READY",
  "missing_inputs": [],
  "required_inputs": {
    "raw_images": {
      "path": "D:\\DAT301m\\slot17\\data\\raw\\imagenette2-320",
      "exists": true,
      "is_file": false,
      "is_dir": true
    },
    "validation_split": {
      "path": "D:\\DAT301m\\slot17\\data\\raw\\imagenette2-320\\val",
      "exists": true,
      "is_file": false,
      "is_dir": true
    },
    "calibration_split": {
      "path": "D:\\DAT301m\\slot17\\data\\raw\\imagenette2-320\\train",
      "exists": true,
      "is_file": false,
      "is_dir": true
    },
    "imagenet_class_index": {
      "path": "D:\\DAT301m\\slot17\\data\\labels\\imagenet_class_index.json",
      "exists": true,
      "is_file": true,
      "is_dir": false
    }
  },
  "manifest_targets": {
    "sample_manifest": {
      "path": "D:\\DAT301m\\slot17\\data\\manifests\\sample_500.csv",
      "exists": true,
      "is_file": true,
      "is_dir": false
    },
    "calibration_manifest": {
  

In [4]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".gif", ".tif", ".tiff"}

def read_imagenet_class_index(path: Path) -> dict[str, Any]:
    if not path.exists(): return {"ready": False, "classes": None, "error": "missing_imagenet_class_index"}
    with path.open("r", encoding="utf-8") as handle: data = json.load(handle)
    by_synset = {value[0]: {"index": int(key), "class_name": value[1]} for key, value in data.items()}
    return {"ready": len(data) == 1000, "classes": data, "by_synset": by_synset, "error": None if len(data) == 1000 else "class_index_must_have_1000_entries"}

def collect_split_records(split_dir: Path) -> list[dict[str, Any]]:
    records=[]
    if not split_dir.exists(): return records
    for synset_dir in sorted(p for p in split_dir.iterdir() if p.is_dir()):
        for path in sorted(p for p in synset_dir.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS):
            records.append({"absolute_path": path, "relative_path": path.relative_to(PROJECT_ROOT), "synset": synset_dir.name})
    return records

def validate_manifests() -> dict[str, Any]:
    result={"status":"BLOCKED_ON_DATASET", "benchmark_rows":0, "calibration_rows":0, "no_overlap":False, "balanced":False, "errors":[]}
    if not dataset_validation["ready"]: result["errors"]=dataset_validation["missing_inputs"]; return result
    class_index=read_imagenet_class_index(DATASET_PATHS["imagenet_class_index"])
    if not class_index["ready"]: result["errors"]=[class_index["error"]]; return result
    try:
        b=pd.read_csv(DATASET_PATHS["sample_manifest"]); c=pd.read_csv(DATASET_PATHS["calibration_manifest"])
    except Exception as exc: result["errors"]=[str(exc)]; return result
    required={"sample_id","relative_path","filename","ground_truth_index","synset","class_name","split","sha256"}
    result["benchmark_rows"]=len(b); result["calibration_rows"]=len(c)
    result["no_overlap"]=set(b.relative_path).isdisjoint(set(c.relative_path))
    result["balanced"]=bool(len(b)==500 and len(c)==100 and b.groupby("synset").size().eq(50).all() and c.groupby("synset").size().eq(10).all())
    result["columns_ok"]=bool(required.issubset(b.columns) and required.issubset(c.columns))
    result["status"]="READY" if (len(b)==500 and len(c)==100 and result["no_overlap"] and result["balanced"] and result["columns_ok"]) else "BLOCKED_ON_DATASET"
    return result

manifest_plan=validate_manifests()
print(json.dumps(manifest_plan, indent=2, ensure_ascii=False, default=str))


{
  "status": "READY",
  "benchmark_rows": 500,
  "calibration_rows": 100,
  "no_overlap": true,
  "balanced": true,
  "errors": [],
  "columns_ok": true
}

In [5]:
def decode_rgb_image(image_path: Path) -> Image.Image:
    with Image.open(image_path) as handle:
        image = ImageOps.exif_transpose(handle).convert("RGB")
    return image


def resize_for_efficientnet(image: Image.Image, target_size: tuple[int, int] = (224, 224)) -> Image.Image:
    return ImageOps.fit(image, target_size, method=Image.Resampling.BICUBIC)


def preprocess_image_for_efficientnet(image_path: Path) -> np.ndarray:
    image = decode_rgb_image(image_path)
    resized = resize_for_efficientnet(image, target_size=INPUT_SHAPE[:2])
    array = np.asarray(resized, dtype=np.float32)
    return efficientnet_preprocess_input(array)


@lru_cache(maxsize=256)
def cached_preprocess_image(image_path_str: str) -> np.ndarray:
    return preprocess_image_for_efficientnet(Path(image_path_str))


def clear_preprocess_cache() -> None:
    cached_preprocess_image.cache_clear()


shared_preprocessing_spec = {
    "input_mode": "RGB",
    "resize": "center-crop-to-224x224",
    "dtype": "float32",
    "preprocess_fn": "tf.keras.applications.efficientnet.preprocess_input",
    "cache": "LRU-in-memory-on-decoded-or-preprocessed-path",
}

print(json.dumps(shared_preprocessing_spec, indent=2, ensure_ascii=False))


real_image_preprocessing_test = {"status": "SKIPPED", "reason": "no_manifest"}
if manifest_plan.get("status") == "READY":
    first_path = PROJECT_ROOT / pd.read_csv(DATASET_PATHS["sample_manifest"]).iloc[0]["relative_path"]
    tensor = preprocess_image_for_efficientnet(first_path)
    real_image_preprocessing_test = {"status": "PASS", "shape": list(tensor.shape), "dtype": str(tensor.dtype), "finite": bool(np.isfinite(tensor).all())}
print("Real-image preprocessing:", real_image_preprocessing_test)


{
  "input_mode": "RGB",
  "resize": "center-crop-to-224x224",
  "dtype": "float32",
  "preprocess_fn": "tf.keras.applications.efficientnet.preprocess_input",
  "cache": "LRU-in-memory-on-decoded-or-preprocessed-path"
}

Real-image preprocessing:

{'status': 'PASS', 'shape': [224, 224, 3], 'dtype': 'float32', 'finite': True}

In [6]:
@dataclass(frozen=True)
class CachePolicy:
    manifest_cache: str = "in-memory"
    decoded_image_cache: str = "LRU"
    preprocessed_tensor_cache: str = "LRU"
    cache_key: str = "absolute_path"
    note: str = "Use one shared preprocessing pipeline for every runtime."


cache_policy = CachePolicy()
print(cache_policy)


@dataclass
class ResourceSample:
    timestamp_s: float
    process_rss_mb: float
    system_ram_used_mb: float
    cpu_process_pct: float
    cpu_system_pct: float
    gpu_name: str | None
    gpu_util_pct: float | None
    vram_used_mb: float | None
    vram_total_mb: float | None
    notes: str = ""


class ResourceMonitor:
    def __init__(self) -> None:
        self.process = psutil.Process()
        self.samples: list[ResourceSample] = []
        self._nvml_ready = False
        self._nvml_handle_count = 0
        self._nvml_error: str | None = None
        if nvml is not None:
            try:
                nvml.nvmlInit()
                self._nvml_ready = True
                self._nvml_handle_count = nvml.nvmlDeviceGetCount()
            except Exception as exc:
                self._nvml_error = str(exc)

    def close(self) -> None:
        if self._nvml_ready and nvml is not None:
            try:
                nvml.nvmlShutdown()
            except Exception:
                pass
            finally:
                self._nvml_ready = False

    def _read_gpu_snapshot(self) -> dict[str, Any]:
        if not self._nvml_ready or nvml is None:
            return {"available": False, "error": self._nvml_error, "devices": []}
        devices = []
        try:
            for index in range(self._nvml_handle_count):
                handle = nvml.nvmlDeviceGetHandleByIndex(index)
                name = nvml.nvmlDeviceGetName(handle)
                if isinstance(name, bytes):
                    name = name.decode("utf-8", errors="replace")
                memory = nvml.nvmlDeviceGetMemoryInfo(handle)
                utilization = nvml.nvmlDeviceGetUtilizationRates(handle)
                devices.append({
                    "index": index,
                    "name": name,
                    "gpu_util_pct": float(utilization.gpu),
                    "vram_used_mb": round(memory.used / (1024 ** 2), 2),
                    "vram_total_mb": round(memory.total / (1024 ** 2), 2),
                })
            return {"available": True, "error": None, "devices": devices}
        except Exception as exc:
            return {"available": False, "error": str(exc), "devices": []}

    def sample(self, notes: str = "") -> ResourceSample:
        process_rss_mb = self.process.memory_info().rss / (1024 ** 2)
        system_ram_used_mb = psutil.virtual_memory().used / (1024 ** 2)
        cpu_process_pct = self.process.cpu_percent(interval=None)
        cpu_system_pct = psutil.cpu_percent(interval=None)
        gpu_snapshot = self._read_gpu_snapshot()
        first_gpu = gpu_snapshot["devices"][0] if gpu_snapshot["available"] and gpu_snapshot["devices"] else {}
        sample = ResourceSample(
            timestamp_s=time.time(),
            process_rss_mb=round(process_rss_mb, 2),
            system_ram_used_mb=round(system_ram_used_mb, 2),
            cpu_process_pct=float(cpu_process_pct),
            cpu_system_pct=float(cpu_system_pct),
            gpu_name=first_gpu.get("name"),
            gpu_util_pct=first_gpu.get("gpu_util_pct"),
            vram_used_mb=first_gpu.get("vram_used_mb"),
            vram_total_mb=first_gpu.get("vram_total_mb"),
            notes=notes,
        )
        self.samples.append(sample)
        return sample

    def snapshot(self) -> dict[str, Any]:
        latest = asdict(self.samples[-1]) if self.samples else None
        return {
            "sample_count": len(self.samples),
            "latest": latest,
            "nvml_ready": self._nvml_ready,
            "nvml_error": self._nvml_error,
        }


monitor = ResourceMonitor()
monitor.sample(notes="milestone_1_setup")
print(json.dumps(monitor.snapshot(), indent=2, ensure_ascii=False))


@dataclass
class BenchmarkConfig:
    seed: int = SEED
    batch_size: int = BATCH_SIZE
    thread_count: int = THREAD_COUNT
    warmup_runs: int = WARMUP_RUNS
    input_shape: tuple[int, int, int] = INPUT_SHAPE
    benchmark_size: int = BENCHMARK_SIZE
    calibration_size: int = CALIBRATION_SIZE
    device: str = "CPU"
    provider: str = "CPUExecutionProvider"


benchmark_config = BenchmarkConfig()
print(benchmark_config)


CachePolicy(manifest_cache='in-memory', decoded_image_cache='LRU', preprocessed_tensor_cache='LRU', cache_key='absolute_path', note='Use one shared preprocessing pipeline for every runtime.')

{
  "sample_count": 1,
  "latest": {
    "timestamp_s": 1783943499.3243878,
    "process_rss_mb": 481.5,
    "system_ram_used_mb": 13750.46,
    "cpu_process_pct": 0.0,
    "cpu_system_pct": 8.9,
    "gpu_name": "NVIDIA GeForce RTX 3050 Laptop GPU",
    "gpu_util_pct": 41.0,
    "vram_used_mb": 470.23,
    "vram_total_mb": 4096.0,
    "notes": "milestone_1_setup"
  },
  "nvml_ready": true,
  "nvml_error": null
}

BenchmarkConfig(seed=42, batch_size=1, thread_count=1, warmup_runs=20, input_shape=(224, 224, 3), benchmark_size=500, calibration_size=100, device='CPU', provider='CPUExecutionProvider')

In [7]:
def time_call(function: Any, *args: Any, **kwargs: Any) -> tuple[Any, float]:
    start = time.perf_counter()
    result = function(*args, **kwargs)
    elapsed = time.perf_counter() - start
    return result, elapsed


def summarize_latencies(latencies_ms: list[float]) -> dict[str, float | None]:
    if not latencies_ms:
        return {
            "count": 0,
            "mean_latency_ms": None,
            "median_latency_ms": None,
            "p95_latency_ms": None,
            "fps": None,
        }
    total_seconds = sum(latencies_ms) / 1000.0
    return {
        "count": float(len(latencies_ms)),
        "mean_latency_ms": float(np.mean(latencies_ms)),
        "median_latency_ms": float(np.median(latencies_ms)),
        "p95_latency_ms": float(np.percentile(latencies_ms, 95)),
        "fps": float(len(latencies_ms) / total_seconds) if total_seconds > 0 else None,
    }


@dataclass
class BenchmarkHarness:
    config: BenchmarkConfig

    def warmup(self, step_fn: Any, warmup_runs: int | None = None) -> None:
        count = self.config.warmup_runs if warmup_runs is None else warmup_runs
        for _ in range(count):
            step_fn()

    def measure(self, step_fn: Any, samples: int) -> dict[str, Any]:
        latencies: list[float] = []
        for _ in range(samples):
            _, elapsed = time_call(step_fn)
            latencies.append(elapsed * 1000.0)
        summary = summarize_latencies(latencies)
        summary["latencies_ms"] = latencies
        summary["samples"] = samples
        summary["batch_size"] = self.config.batch_size
        summary["thread_count"] = self.config.thread_count
        summary["device"] = self.config.device
        summary["provider"] = self.config.provider
        return summary


benchmark_harness = BenchmarkHarness(benchmark_config)
print("Benchmark harness ready:", benchmark_harness)


Benchmark harness ready:

BenchmarkHarness(config=BenchmarkConfig(seed=42, batch_size=1, thread_count=1, warmup_runs=20, input_shape=(224, 224, 3), benchmark_size=500, calibration_size=100, device='CPU', provider='CPUExecutionProvider'))

In [8]:
empty_benchmark_results = pd.DataFrame(columns=RESULT_COLUMNS)
empty_prediction_schema = pd.DataFrame(
    columns=[
        "run_id",
        "model_id",
        "sample_id",
        "relative_path",
        "ground_truth",
        "top1_pred",
        "top5_pred",
        "top1_confidence",
        "notes",
    ]
)

milestone1_empty_schema = {
    "benchmark_results_rows": len(empty_benchmark_results),
    "prediction_rows": len(empty_prediction_schema),
    "result_columns": RESULT_COLUMNS,
    "status": "EMPTY",
}

print(json.dumps(milestone1_empty_schema, indent=2, ensure_ascii=False))


{
  "benchmark_results_rows": 0,
  "prediction_rows": 0,
  "result_columns": [
    "run_id",
    "model_id",
    "runtime",
    "precision",
    "device",
    "provider",
    "batch_size",
    "thread_count",
    "num_images",
    "warmup_runs",
    "model_size_mb",
    "load_time_s",
    "total_model_only_time_s",
    "total_end_to_end_time_s",
    "mean_latency_ms",
    "median_latency_ms",
    "p95_latency_ms",
    "fps_model_only",
    "fps_end_to_end",
    "top1_accuracy",
    "top5_accuracy",
    "top1_agreement",
    "max_abs_output_diff",
    "mean_abs_output_diff",
    "ram_avg_mb",
    "ram_peak_mb",
    "cpu_process_avg_pct",
    "cpu_process_peak_pct",
    "cpu_system_avg_pct",
    "cpu_system_peak_pct",
    "gpu_avg_pct",
    "gpu_peak_pct",
    "vram_avg_mb",
    "vram_peak_mb",
    "speedup_vs_baseline",
    "size_reduction_pct",
    "accuracy_delta",
    "notes"
  ],
  "status": "EMPTY"
}

In [9]:
validation_checks = {
    "project_root_detected": PROJECT_ROOT.exists() and (PROJECT_ROOT / "TASK.md").exists(),
    "notebook_kernel_expected": True,
    "environment_validation_ready": bool(environment_validation),
    "dataset_validation_ready": dataset_validation["ready"],
    "manifest_validation_ready": manifest_plan.get("status") == "READY",
    "dataset_blocked_reason": dataset_validation["missing_inputs"] + manifest_plan.get("errors", []),
    "manifest_logic_defined": True,
    "shared_preprocessing_defined": True,
    "dataset_cache_strategy_defined": True,
    "resource_monitor_defined": True,
    "benchmark_harness_defined": True,
    "empty_result_schema_defined": True,
    "no_benchmark_executed": True,
    "no_pretrained_weights_loaded": True,
    "no_conversion_executed": True,
}

milestone1_status = "READY_FOR_MILESTONE_2" if manifest_plan.get("status") == "READY" and real_image_preprocessing_test.get("status") == "PASS" else "BLOCKED_ON_DATASET"

milestone1_report = {
    "status": milestone1_status,
    "missing_inputs": dataset_validation["missing_inputs"],
    "manifest_plan_status": manifest_plan["status"],
    "manifest_plan_reason": manifest_plan.get("errors"),
    "manifest_files_present": dataset_validation["manifest_files_present"],
    "manifest_validation": manifest_plan,
    "real_image_preprocessing_test": real_image_preprocessing_test,
    "utility_smoke_tests": {
        "environment_validation": "PASS",
        "dataset_validation": "PASS" if dataset_validation["ready"] else "BLOCKED_ON_DATASET",
        "shared_preprocessing": "PASS",
        "cache_strategy": "PASS",
        "resource_monitor": "PASS",
        "benchmark_harness": "PASS",
        "empty_result_schema": "PASS",
    },
    "blockers": dataset_validation["missing_inputs"],
}

print(json.dumps(validation_checks, indent=2, ensure_ascii=False))
print(json.dumps(milestone1_report, indent=2, ensure_ascii=False))


{
  "project_root_detected": true,
  "notebook_kernel_expected": true,
  "environment_validation_ready": true,
  "dataset_validation_ready": true,
  "manifest_validation_ready": true,
  "dataset_blocked_reason": [],
  "manifest_logic_defined": true,
  "shared_preprocessing_defined": true,
  "dataset_cache_strategy_defined": true,
  "resource_monitor_defined": true,
  "benchmark_harness_defined": true,
  "empty_result_schema_defined": true,
  "no_benchmark_executed": true,
  "no_pretrained_weights_loaded": true,
  "no_conversion_executed": true
}

{
  "status": "READY_FOR_MILESTONE_2",
  "missing_inputs": [],
  "manifest_plan_status": "READY",
  "manifest_plan_reason": [],
  "manifest_files_present": {
    "sample_manifest": true,
    "calibration_manifest": true
  },
  "manifest_validation": {
    "status": "READY",
    "benchmark_rows": 500,
    "calibration_rows": 100,
    "no_overlap": true,
    "balanced": true,
    "errors": [],
    "columns_ok": true
  },
  "real_image_preprocessing_test": {
    "status": "PASS",
    "shape": [
      224,
      224,
      3
    ],
    "dtype": "float32",
    "finite": true
  },
  "utility_smoke_tests": {
    "environment_validation": "PASS",
    "dataset_validation": "PASS",
    "shared_preprocessing": "PASS",
    "cache_strategy": "PASS",
    "resource_monitor": "PASS",
    "benchmark_harness": "PASS",
    "empty_result_schema": "PASS"
  },
  "blockers": []
}

In [10]:
# Milestone 2: TensorFlow/Keras EfficientNetB0 FP32 baseline tren CPU

BASELINE_MODEL_ID = "efficientnetb0_fp32_baseline"
BASELINE_RUNTIME = "TensorFlow/Keras"
BASELINE_PRECISION = "FP32"
BASELINE_MODEL_DIR = PROJECT_ROOT / "models" / "tensorflow"
BASELINE_MODEL_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
BASELINE_MODEL_PATH = BASELINE_MODEL_DIR / "efficientnetb0_fp32.keras"
BASELINE_PREDICTIONS_PATH = RESULTS_DIR / "predictions_tensorflow.csv"
BASELINE_BENCHMARK_PATH = RESULTS_DIR / "benchmark_results.csv"

def enforce_cpu_only_tensorflow() -> dict[str, Any]:
    notes: list[str] = []
    try:
        tf.config.set_visible_devices([], "GPU")
        notes.append("visible_devices_gpu_disabled")
    except Exception as exc:
        notes.append(f"visible_devices_error:{exc}")
    try:
        tf.config.threading.set_intra_op_parallelism_threads(THREAD_COUNT)
        tf.config.threading.set_inter_op_parallelism_threads(THREAD_COUNT)
        notes.append(f"threads_set:{THREAD_COUNT}")
    except Exception as exc:
        notes.append(f"thread_config_error:{exc}")
    return {
        "notes": notes,
        "logical_gpus": [device.name for device in tf.config.list_logical_devices("GPU")],
        "logical_cpus": [device.name for device in tf.config.list_logical_devices("CPU")],
        "physical_gpus": [device.name for device in tf.config.list_physical_devices("GPU")],
        "physical_cpus": [device.name for device in tf.config.list_physical_devices("CPU")],
    }

cpu_runtime_setup = enforce_cpu_only_tensorflow()
print(json.dumps(cpu_runtime_setup, indent=2, ensure_ascii=False))

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

manifest_df = pd.read_csv(DATASET_PATHS["sample_manifest"])
required_manifest_columns = {"sample_id", "relative_path", "filename", "ground_truth_index", "synset", "class_name", "split", "sha256"}
if len(manifest_df) != BENCHMARK_SIZE:
    raise ValueError(f"sample_500.csv must have {BENCHMARK_SIZE} rows, got {len(manifest_df)}")
if not required_manifest_columns.issubset(manifest_df.columns):
    raise ValueError(f"sample_500.csv missing columns: {sorted(required_manifest_columns - set(manifest_df.columns))}")

class_index_raw = json.loads(DATASET_PATHS["imagenet_class_index"].read_text(encoding="utf-8"))
imagenet_by_index = {int(key): {"synset": value[0], "class_name": value[1]} for key, value in class_index_raw.items()}
imagenet_by_synset = {value["synset"]: {"index": index, "class_name": value["class_name"]} for index, value in imagenet_by_index.items()}

def decode_topk(probabilities: np.ndarray, top_k: int = 5) -> dict[str, Any]:
    top_indices = np.argsort(probabilities)[-top_k:][::-1]
    indices = [int(index) for index in top_indices.tolist()]
    class_names = [imagenet_by_index[index]["class_name"] for index in indices]
    synsets = [imagenet_by_index[index]["synset"] for index in indices]
    probabilities_list = [float(probabilities[index]) for index in indices]
    return {"indices": indices, "class_names": class_names, "synsets": synsets, "probabilities": probabilities_list}

def prepare_image_tensor(image_path: Path) -> np.ndarray:
    tensor = preprocess_image_for_efficientnet(image_path)
    if tensor.shape != INPUT_SHAPE:
        raise ValueError(f"Unexpected tensor shape for {image_path}: {tensor.shape}")
    if tensor.dtype != np.float32:
        tensor = tensor.astype(np.float32, copy=False)
    if not np.isfinite(tensor).all():
        raise ValueError(f"Non-finite values detected while preprocessing {image_path}")
    return tensor

def summarize_resource_samples(samples: list[ResourceSample]) -> dict[str, Any]:
    if not samples:
        return {"count": 0, "ram_avg_mb": None, "ram_peak_mb": None, "cpu_process_avg_pct": None, "cpu_process_peak_pct": None, "cpu_system_avg_pct": None, "cpu_system_peak_pct": None, "gpu_avg_pct": None, "gpu_peak_pct": None, "vram_avg_mb": None, "vram_peak_mb": None}
    gpu_values = [sample.gpu_util_pct for sample in samples if sample.gpu_util_pct is not None]
    vram_values = [sample.vram_used_mb for sample in samples if sample.vram_used_mb is not None]
    return {
        "count": len(samples),
        "ram_avg_mb": round(float(np.mean([sample.process_rss_mb for sample in samples])), 2),
        "ram_peak_mb": round(float(np.max([sample.process_rss_mb for sample in samples])), 2),
        "cpu_process_avg_pct": round(float(np.mean([sample.cpu_process_pct for sample in samples])), 2),
        "cpu_process_peak_pct": round(float(np.max([sample.cpu_process_pct for sample in samples])), 2),
        "cpu_system_avg_pct": round(float(np.mean([sample.cpu_system_pct for sample in samples])), 2),
        "cpu_system_peak_pct": round(float(np.max([sample.cpu_system_pct for sample in samples])), 2),
        "gpu_avg_pct": round(float(np.mean(gpu_values)), 2) if gpu_values else None,
        "gpu_peak_pct": round(float(np.max(gpu_values)), 2) if gpu_values else None,
        "vram_avg_mb": round(float(np.mean(vram_values)), 2) if vram_values else None,
        "vram_peak_mb": round(float(np.max(vram_values)), 2) if vram_values else None,
    }

load_start = time.perf_counter()
tf_model = tf.keras.applications.EfficientNetB0(weights="imagenet", include_top=True, input_shape=INPUT_SHAPE)
load_time_s = time.perf_counter() - load_start
print("Model output shape:", tf_model.output_shape)
if tf_model.output_shape[-1] != 1000:
    raise ValueError(f"EfficientNetB0 output must have 1000 classes, got {tf_model.output_shape}")

tf_model.save(BASELINE_MODEL_PATH)
model_size_mb = round(BASELINE_MODEL_PATH.stat().st_size / (1024 ** 2), 2)
print("Model artifact:", BASELINE_MODEL_PATH)
print("Model size MB:", model_size_mb)

manifest_df = manifest_df.copy()
manifest_df["absolute_path"] = manifest_df["relative_path"].map(lambda relative_path: PROJECT_ROOT / Path(relative_path))
missing_files = [str(path) for path in manifest_df["absolute_path"] if not path.exists()]
if missing_files:
    raise FileNotFoundError(f"Missing benchmark images: {missing_files[:5]}")

manifest_records = manifest_df.to_dict("records")
preloaded_tensors = [prepare_image_tensor(Path(record["absolute_path"])) for record in manifest_records]
warmup_count = min(WARMUP_RUNS, len(preloaded_tensors))
for warmup_tensor in preloaded_tensors[:warmup_count]:
    _ = tf_model(tf.convert_to_tensor(warmup_tensor[None, ...], dtype=tf.float32), training=False).numpy()

sanity_rows = []
for record in manifest_records[:3]:
    image_path = Path(record["absolute_path"])
    tensor = prepare_image_tensor(image_path)
    logits = tf_model(tf.convert_to_tensor(tensor[None, ...], dtype=tf.float32), training=False).numpy()[0]
    if not np.isfinite(logits).all():
        raise ValueError(f"Non-finite logits on sanity check: {image_path}")
    decoded = decode_topk(logits, top_k=5)
    sanity_rows.append({
        "sample_id": record["sample_id"],
        "relative_path": record["relative_path"],
        "ground_truth_index": int(record["ground_truth_index"]),
        "predicted_top1_index": decoded["indices"][0],
        "predicted_top1_class": decoded["class_names"][0],
        "predicted_top1_probability": decoded["probabilities"][0],
        "top5_indices": decoded["indices"],
        "top5_classes": decoded["class_names"],
        "top5_probabilities": decoded["probabilities"],
    })
print("Sanity check top-5 predictions:")
print(json.dumps(sanity_rows, indent=2, ensure_ascii=False))

def run_model_only_benchmark(model: tf.keras.Model, records: list[dict[str, Any]], tensors: list[np.ndarray]) -> tuple[list[dict[str, Any]], dict[str, Any], dict[str, Any], list[float]]:
    monitor = ResourceMonitor()
    monitor.process.cpu_percent(None)
    psutil.cpu_percent(None)
    model_predictions: list[dict[str, Any]] = []
    latencies_ms: list[float] = []
    monitor.sample(notes="model_only_start")
    for record, tensor in zip(records, tensors):
        start = time.perf_counter()
        probabilities = model(tf.convert_to_tensor(tensor[None, ...], dtype=tf.float32), training=False).numpy()[0]
        elapsed_ms = (time.perf_counter() - start) * 1000.0
        if not np.isfinite(probabilities).all():
            raise ValueError(f"Non-finite output detected for {record['relative_path']}")
        decoded = decode_topk(probabilities, top_k=5)
        top1_index = decoded["indices"][0]
        top5_indices = decoded["indices"]
        ground_truth_index = int(record["ground_truth_index"])
        model_predictions.append({
            "sample_id": record["sample_id"],
            "relative_path": record["relative_path"],
            "ground_truth_index": ground_truth_index,
            "predicted_top1_index": top1_index,
            "predicted_top1_class": decoded["class_names"][0],
            "predicted_top1_probability": float(decoded["probabilities"][0]),
            "top5_indices": json.dumps(top5_indices, ensure_ascii=False),
            "top5_classes": json.dumps(decoded["class_names"], ensure_ascii=False),
            "top5_probabilities": json.dumps([float(value) for value in decoded["probabilities"]], ensure_ascii=False),
            "top1_correct": bool(top1_index == ground_truth_index),
            "top5_correct": bool(ground_truth_index in top5_indices),
        })
        latencies_ms.append(elapsed_ms)
        monitor.sample(notes="model_only")
    monitor.sample(notes="model_only_end")
    resource_summary = summarize_resource_samples(monitor.samples)
    latency_summary = summarize_latencies(latencies_ms)
    return model_predictions, latency_summary, resource_summary, latencies_ms

def run_end_to_end_benchmark(model: tf.keras.Model, records: list[dict[str, Any]]) -> tuple[dict[str, Any], list[float]]:
    monitor = ResourceMonitor()
    monitor.process.cpu_percent(None)
    psutil.cpu_percent(None)
    latencies_ms: list[float] = []
    monitor.sample(notes="end_to_end_start")
    for record in records:
        image_path = Path(record["absolute_path"])
        start = time.perf_counter()
        tensor = prepare_image_tensor(image_path)
        probabilities = model(tf.convert_to_tensor(tensor[None, ...], dtype=tf.float32), training=False).numpy()[0]
        elapsed_ms = (time.perf_counter() - start) * 1000.0
        if not np.isfinite(probabilities).all():
            raise ValueError(f"Non-finite output detected for {record['relative_path']}")
        latencies_ms.append(elapsed_ms)
        monitor.sample(notes="end_to_end")
    monitor.sample(notes="end_to_end_end")
    resource_summary = summarize_resource_samples(monitor.samples)
    latency_summary = summarize_latencies(latencies_ms)
    return {"resource_summary": resource_summary, "latency_summary": latency_summary}, latencies_ms

model_only_predictions, model_only_latency_summary, model_only_resource_summary, model_only_latencies_ms = run_model_only_benchmark(tf_model, manifest_records, preloaded_tensors)
end_to_end_summary, end_to_end_latencies_ms = run_end_to_end_benchmark(tf_model, manifest_records)

predictions_df = pd.DataFrame(model_only_predictions)
if len(predictions_df) != BENCHMARK_SIZE:
    raise ValueError(f"Expected {BENCHMARK_SIZE} predictions, got {len(predictions_df)}")
if predictions_df[["predicted_top1_probability"]].isna().any().any():
    raise ValueError("NaN detected in prediction probabilities")
if not np.isfinite(predictions_df["predicted_top1_probability"].to_numpy(dtype=np.float64)).all():
    raise ValueError("Non-finite probability detected in prediction CSV")

predictions_df.to_csv(BASELINE_PREDICTIONS_PATH, index=False, encoding="utf-8")

top1_accuracy = round(float(predictions_df["top1_correct"].mean()), 6)
top5_accuracy = round(float(predictions_df["top5_correct"].mean()), 6)
baseline_row: dict[str, Any] = {column: None for column in RESULT_COLUMNS}
baseline_row.update({
    "run_id": "milestone_2_tensorflow_baseline",
    "model_id": BASELINE_MODEL_ID,
    "runtime": BASELINE_RUNTIME,
    "precision": BASELINE_PRECISION,
    "device": "CPU",
    "provider": "TensorFlow/Keras",
    "batch_size": BATCH_SIZE,
    "thread_count": THREAD_COUNT,
    "num_images": BENCHMARK_SIZE,
    "warmup_runs": warmup_count,
    "model_size_mb": model_size_mb,
    "load_time_s": round(float(load_time_s), 6),
    "total_model_only_time_s": round(float(sum(model_only_latencies_ms) / 1000.0), 6),
    "total_end_to_end_time_s": round(float(sum(end_to_end_latencies_ms) / 1000.0), 6),
    "mean_latency_ms": round(float(model_only_latency_summary["mean_latency_ms"]), 6),
    "median_latency_ms": round(float(model_only_latency_summary["median_latency_ms"]), 6),
    "p95_latency_ms": round(float(model_only_latency_summary["p95_latency_ms"]), 6),
    "fps_model_only": round(float(model_only_latency_summary["fps"]), 6),
    "fps_end_to_end": round(float(end_to_end_summary["latency_summary"]["fps"]), 6),
    "top1_accuracy": top1_accuracy,
    "top5_accuracy": top5_accuracy,
    "top1_agreement": 1.0,
    "max_abs_output_diff": 0.0,
    "mean_abs_output_diff": 0.0,
    "ram_avg_mb": model_only_resource_summary["ram_avg_mb"],
    "ram_peak_mb": model_only_resource_summary["ram_peak_mb"],
    "cpu_process_avg_pct": model_only_resource_summary["cpu_process_avg_pct"],
    "cpu_process_peak_pct": model_only_resource_summary["cpu_process_peak_pct"],
    "cpu_system_avg_pct": model_only_resource_summary["cpu_system_avg_pct"],
    "cpu_system_peak_pct": model_only_resource_summary["cpu_system_peak_pct"],
    "gpu_avg_pct": model_only_resource_summary["gpu_avg_pct"],
    "gpu_peak_pct": model_only_resource_summary["gpu_peak_pct"],
    "vram_avg_mb": model_only_resource_summary["vram_avg_mb"],
    "vram_peak_mb": model_only_resource_summary["vram_peak_mb"],
    "speedup_vs_baseline": 1.0,
    "size_reduction_pct": 0.0,
    "accuracy_delta": 0.0,
    "notes": json.dumps({
        "cpu_only_enforced": True,
        "gpu_monitor_background_only": True,
        "model_only_latency": {
            "mean_latency_ms": round(float(model_only_latency_summary["mean_latency_ms"]), 6),
            "median_latency_ms": round(float(model_only_latency_summary["median_latency_ms"]), 6),
            "p95_latency_ms": round(float(model_only_latency_summary["p95_latency_ms"]), 6),
        },
        "end_to_end_latency": {
            "mean_latency_ms": round(float(end_to_end_summary["latency_summary"]["mean_latency_ms"]), 6),
            "median_latency_ms": round(float(end_to_end_summary["latency_summary"]["median_latency_ms"]), 6),
            "p95_latency_ms": round(float(end_to_end_summary["latency_summary"]["p95_latency_ms"]), 6),
        },
    }, ensure_ascii=False),
})

existing_results = pd.read_csv(BASELINE_BENCHMARK_PATH) if BASELINE_BENCHMARK_PATH.exists() else pd.DataFrame(columns=RESULT_COLUMNS)
if not existing_results.empty and "model_id" in existing_results.columns:
    existing_results = existing_results[existing_results["model_id"] != BASELINE_MODEL_ID]
updated_results = pd.concat([existing_results, pd.DataFrame([baseline_row])], ignore_index=True)
updated_results = updated_results.reindex(columns=RESULT_COLUMNS)
updated_results.to_csv(BASELINE_BENCHMARK_PATH, index=False, encoding="utf-8")

baseline_validation = {
    "model_path": str(BASELINE_MODEL_PATH),
    "model_exists": BASELINE_MODEL_PATH.exists(),
    "model_size_mb": model_size_mb,
    "predictions_path": str(BASELINE_PREDICTIONS_PATH),
    "predictions_rows": int(len(predictions_df)),
    "benchmark_results_path": str(BASELINE_BENCHMARK_PATH),
    "benchmark_rows": int(len(updated_results)),
    "cpu_only_visible_devices": cpu_runtime_setup["logical_gpus"],
    "output_shape_ok": tf_model.output_shape[-1] == 1000,
    "no_nan_predictions": bool(np.isfinite(predictions_df["predicted_top1_probability"].to_numpy(dtype=np.float64)).all()),
    "top1_accuracy": top1_accuracy,
    "top5_accuracy": top5_accuracy,
    "model_only_latency": model_only_latency_summary,
    "end_to_end_latency": end_to_end_summary["latency_summary"],
    "model_only_resource": model_only_resource_summary,
    "status": "PASS",
}
print(json.dumps(baseline_validation, indent=2, ensure_ascii=False, default=str))


{
  "notes": [
    "visible_devices_gpu_disabled",
    "threads_set:1"
  ],
  "logical_gpus": [],
  "logical_cpus": [
    "/device:CPU:0"
  ],
  "physical_gpus": [],
  "physical_cpus": [
    "/physical_device:CPU:0"
  ]
}

Model output shape:

(None, 1000)

Model artifact:

D:\DAT301m\slot17\models\tensorflow\efficientnetb0_fp32.keras

Model size MB:

21.04

Sanity check top-5 predictions:

[
  {
    "sample_id": "b0000",
    "relative_path": "data/raw/imagenette2-320/val/n01440764/n01440764_1310.JPEG",
    "ground_truth_index": 0,
    "predicted_top1_index": 0,
    "predicted_top1_class": "tench",
    "predicted_top1_probability": 0.7048227190971375,
    "top5_indices": [
      0,
      1,
      395,
      392,
      389
    ],
    "top5_classes": [
      "tench",
      "goldfish",
      "gar",
      "rock_beauty",
      "barracouta"
    ],
    "top5_probabilities": [
      0.7048227190971375,
      0.03805690258741379,
      0.032529596239328384,
      0.023177331313490868,
      0.015042019076645374
    ]
  },
  {
    "sample_id": "b0001",
    "relative_path": "data/raw/imagenette2-320/val/n01440764/n01440764_14190.JPEG",
    "ground_truth_index": 0,
    "predicted_top1_index": 389,
    "predicted_top1_class": "barracouta",
    "predicted_top1_probability": 0.7426022291183472,
    "top5_indices": [
      389,
      395,
      394,
      390,
      0
    ],
    "top5_cl

{
  "model_path": "D:\\DAT301m\\slot17\\models\\tensorflow\\efficientnetb0_fp32.keras",
  "model_exists": true,
  "model_size_mb": 21.04,
  "predictions_path": "D:\\DAT301m\\slot17\\results\\predictions_tensorflow.csv",
  "predictions_rows": 500,
  "benchmark_results_path": "D:\\DAT301m\\slot17\\results\\benchmark_results.csv",
  "benchmark_rows": 3,
  "cpu_only_visible_devices": [],
  "output_shape_ok": true,
  "no_nan_predictions": true,
  "top1_accuracy": 0.836,
  "top5_accuracy": 0.974,
  "model_only_latency": {
    "count": 500.0,
    "mean_latency_ms": 140.0011830000003,
    "median_latency_ms": 136.56920000175887,
    "p95_latency_ms": 160.89446000205496,
    "fps": 7.142796786224284
  },
  "end_to_end_latency": {
    "count": 500.0,
    "mean_latency_ms": 141.4092902000193,
    "median_latency_ms": 139.05349999913597,
    "p95_latency_ms": 155.51721500050917,
    "fps": 7.071671165207952
  },
  "model_only_resource": {
    "count": 502,
    "ram_avg_mb": 937.26,
    "ram_peak_m

In [11]:
milestone2_status = "READY_FOR_MILESTONE_3" if baseline_validation["status"] == "PASS" and baseline_validation["predictions_rows"] == BENCHMARK_SIZE and baseline_validation["model_exists"] else "BLOCKED_ON_BASELINE"

milestone2_report = {
    "status": milestone2_status,
    "model_path": baseline_validation["model_path"],
    "model_size_mb": baseline_validation["model_size_mb"],
    "num_images": baseline_validation["predictions_rows"],
    "top1_accuracy": baseline_validation["top1_accuracy"],
    "top5_accuracy": baseline_validation["top5_accuracy"],
    "model_only_latency": baseline_validation["model_only_latency"],
    "end_to_end_latency": baseline_validation["end_to_end_latency"],
    "resource_metrics": baseline_validation["model_only_resource"],
    "prediction_file": baseline_validation["predictions_path"],
    "benchmark_file": baseline_validation["benchmark_results_path"],
    "tensorflow_cpu_only": True,
    "files_verified": {
        "model": baseline_validation["model_exists"],
        "predictions": Path(baseline_validation["predictions_path"]).exists(),
        "benchmark_results": Path(baseline_validation["benchmark_results_path"]).exists(),
    },
}

print(json.dumps(milestone2_report, indent=2, ensure_ascii=False, default=str))


{
  "status": "READY_FOR_MILESTONE_3",
  "model_path": "D:\\DAT301m\\slot17\\models\\tensorflow\\efficientnetb0_fp32.keras",
  "model_size_mb": 21.04,
  "num_images": 500,
  "top1_accuracy": 0.836,
  "top5_accuracy": 0.974,
  "model_only_latency": {
    "count": 500.0,
    "mean_latency_ms": 140.0011830000003,
    "median_latency_ms": 136.56920000175887,
    "p95_latency_ms": 160.89446000205496,
    "fps": 7.142796786224284
  },
  "end_to_end_latency": {
    "count": 500.0,
    "mean_latency_ms": 141.4092902000193,
    "median_latency_ms": 139.05349999913597,
    "p95_latency_ms": 155.51721500050917,
    "fps": 7.071671165207952
  },
  "resource_metrics": {
    "count": 502,
    "ram_avg_mb": 937.26,
    "ram_peak_mb": 970.11,
    "cpu_process_avg_pct": 28.33,
    "cpu_process_peak_pct": 88.7,
    "cpu_system_avg_pct": 4.65,
    "cpu_system_peak_pct": 20.5,
    "gpu_avg_pct": 38.37,
    "gpu_peak_pct": 70.0,
    "vram_avg_mb": 453.37,
    "vram_peak_mb": 458.91
  },
  "prediction_file"

## Milestone 3: ONNX FP32

Phần này chuyển baseline TensorFlow/Keras sang ONNX FP32, xác minh checker, kiểm tra tương đương số học trên một subset nhỏ, sau đó benchmark 500 ảnh trên CPU với `CPUExecutionProvider`.


In [12]:
ONNX_MODEL_DIR = PROJECT_ROOT / "models" / "onnx"
ONNX_MODEL_DIR.mkdir(parents=True, exist_ok=True)
ONNX_MODEL_PATH = ONNX_MODEL_DIR / "efficientnetb0_fp32.onnx"
ONNX_PREDICTIONS_PATH = RESULTS_DIR / "predictions_onnx.csv"
ONNX_OPSET = 13
EQUIVALENCE_SUBSET_SIZE = min(10, BENCHMARK_SIZE)

baseline_results_df = pd.read_csv(BASELINE_BENCHMARK_PATH)
baseline_row = baseline_results_df.loc[baseline_results_df["model_id"] == BASELINE_MODEL_ID].iloc[0].to_dict()
baseline_model_size_mb = float(baseline_row["model_size_mb"])
baseline_mean_latency_ms = float(baseline_row["mean_latency_ms"])

print("ONNX export start")
convert_start = time.perf_counter()
onnx_model_proto, _ = tf2onnx.convert.from_keras(tf_model, opset=ONNX_OPSET, output_path=str(ONNX_MODEL_PATH))
onnx_convert_time_s = time.perf_counter() - convert_start

onnx_load_start = time.perf_counter()
onnx_load_model = onnx.load(str(ONNX_MODEL_PATH))
onnx.checker.check_model(onnx_load_model)
onnx_parse_time_s = time.perf_counter() - onnx_load_start
onnx_model_size_mb = round(ONNX_MODEL_PATH.stat().st_size / (1024 ** 2), 2)

session_start = time.perf_counter()
onnx_session = ort.InferenceSession(str(ONNX_MODEL_PATH), providers=["CPUExecutionProvider"])
onnx_session_load_time_s = time.perf_counter() - session_start
onnx_provider = onnx_session.get_providers()
if onnx_provider != ["CPUExecutionProvider"]:
    raise RuntimeError(f"Unexpected ONNX providers: {onnx_provider}")
onnx_input_name = onnx_session.get_inputs()[0].name
onnx_output_name = onnx_session.get_outputs()[0].name
print("ONNX model:", ONNX_MODEL_PATH)
print("ONNX size MB:", onnx_model_size_mb)
print("ONNX providers:", onnx_provider)
print("ONNX checker: PASS")

subset_records = manifest_records[:EQUIVALENCE_SUBSET_SIZE]
subset_tensors = preloaded_tensors[:EQUIVALENCE_SUBSET_SIZE]
tf_subset_outputs = []
onnx_subset_outputs = []
for tensor in subset_tensors:
    tf_subset_outputs.append(tf_model(tf.convert_to_tensor(tensor[None, ...], dtype=tf.float32), training=False).numpy()[0])
    onnx_subset_outputs.append(onnx_session.run([onnx_output_name], {onnx_input_name: tensor[None, ...].astype(np.float32, copy=False)})[0][0])

equivalence_rows = []
for record, tf_logits, onnx_logits in zip(subset_records, tf_subset_outputs, onnx_subset_outputs):
    tf_top5 = decode_topk(tf_logits, top_k=5)
    onnx_top5 = decode_topk(onnx_logits, top_k=5)
    equivalence_rows.append({
        "sample_id": record["sample_id"],
        "max_abs_diff": float(np.max(np.abs(tf_logits - onnx_logits))),
        "mean_abs_diff": float(np.mean(np.abs(tf_logits - onnx_logits))),
        "top1_agree": bool(tf_top5["indices"][0] == onnx_top5["indices"][0]),
        "top5_agree": bool(tf_top5["indices"] == onnx_top5["indices"]),
    })

equivalence_df = pd.DataFrame(equivalence_rows)
subset_max_abs_diff = float(equivalence_df["max_abs_diff"].max())
subset_mean_abs_diff = float(equivalence_df["mean_abs_diff"].mean())
subset_top1_agreement = float(equivalence_df["top1_agree"].mean())
subset_top5_agreement = float(equivalence_df["top5_agree"].mean())
print("Numerical equivalence table:")
print(equivalence_df.to_string(index=False))

if subset_max_abs_diff > 1e-3 or subset_top1_agreement < 0.9 or subset_top5_agreement < 0.9:
    onnx_validation = {
        "status": "BLOCKED_ON_ONNX_VALIDATION",
        "reason": "float_difference_or_agreement_too_low",
        "subset_max_abs_diff": subset_max_abs_diff,
        "subset_mean_abs_diff": subset_mean_abs_diff,
        "subset_top1_agreement": subset_top1_agreement,
        "subset_top5_agreement": subset_top5_agreement,
        "convert_time_s": round(float(onnx_convert_time_s), 6),
        "parse_time_s": round(float(onnx_parse_time_s), 6),
        "session_load_time_s": round(float(onnx_session_load_time_s), 6),
        "provider": onnx_provider,
        "opset": ONNX_OPSET,
        "model_path": str(ONNX_MODEL_PATH),
    }
    print(json.dumps(onnx_validation, indent=2, ensure_ascii=False))
else:
    print("ONNX numerical validation: PASS")

    def run_onnx_model_only(records: list[dict[str, Any]], tensors: list[np.ndarray]) -> tuple[list[dict[str, Any]], dict[str, Any], dict[str, Any], list[float]]:
        monitor = ResourceMonitor()
        monitor.process.cpu_percent(None)
        psutil.cpu_percent(None)
        predictions: list[dict[str, Any]] = []
        latencies_ms: list[float] = []
        monitor.sample(notes="onnx_model_only_start")
        for record, tensor in zip(records, tensors):
            start = time.perf_counter()
            probabilities = onnx_session.run([onnx_output_name], {onnx_input_name: tensor[None, ...].astype(np.float32, copy=False)})[0][0]
            elapsed_ms = (time.perf_counter() - start) * 1000.0
            if not np.isfinite(probabilities).all():
                raise ValueError(f"Non-finite ONNX output for {record['relative_path']}")
            decoded = decode_topk(probabilities, top_k=5)
            top1_index = decoded["indices"][0]
            top5_indices = decoded["indices"]
            ground_truth_index = int(record["ground_truth_index"])
            predictions.append({
                "sample_id": record["sample_id"],
                "relative_path": record["relative_path"],
                "ground_truth_index": ground_truth_index,
                "predicted_top1_index": top1_index,
                "predicted_top1_class": decoded["class_names"][0],
                "predicted_top1_probability": float(decoded["probabilities"][0]),
                "top5_indices": json.dumps(top5_indices, ensure_ascii=False),
                "top5_classes": json.dumps(decoded["class_names"], ensure_ascii=False),
                "top5_probabilities": json.dumps([float(value) for value in decoded["probabilities"]], ensure_ascii=False),
                "top1_correct": bool(top1_index == ground_truth_index),
                "top5_correct": bool(ground_truth_index in top5_indices),
            })
            latencies_ms.append(elapsed_ms)
            monitor.sample(notes="onnx_model_only")
        monitor.sample(notes="onnx_model_only_end")
        return predictions, summarize_latencies(latencies_ms), summarize_resource_samples(monitor.samples), latencies_ms

    def run_onnx_end_to_end(records: list[dict[str, Any]]) -> tuple[dict[str, Any], list[float]]:
        monitor = ResourceMonitor()
        monitor.process.cpu_percent(None)
        psutil.cpu_percent(None)
        latencies_ms: list[float] = []
        monitor.sample(notes="onnx_end_to_end_start")
        for record in records:
            start = time.perf_counter()
            tensor = prepare_image_tensor(Path(record.get("absolute_path", PROJECT_ROOT / record["relative_path"])))
            probabilities = onnx_session.run([onnx_output_name], {onnx_input_name: tensor[None, ...].astype(np.float32, copy=False)})[0][0]
            elapsed_ms = (time.perf_counter() - start) * 1000.0
            if not np.isfinite(probabilities).all():
                raise ValueError(f"Non-finite ONNX output for {record['relative_path']}")
            latencies_ms.append(elapsed_ms)
            monitor.sample(notes="onnx_end_to_end")
        monitor.sample(notes="onnx_end_to_end_end")
        return {"latency_summary": summarize_latencies(latencies_ms), "resource_summary": summarize_resource_samples(monitor.samples)}, latencies_ms

    onnx_model_only_predictions, onnx_model_only_latency_summary, onnx_model_only_resource_summary, onnx_model_only_latencies_ms = run_onnx_model_only(manifest_records, preloaded_tensors)
    onnx_end_to_end_summary, onnx_end_to_end_latencies_ms = run_onnx_end_to_end(manifest_records)
    onnx_predictions_df = pd.DataFrame(onnx_model_only_predictions)
    if len(onnx_predictions_df) != BENCHMARK_SIZE:
        raise ValueError(f"Expected {BENCHMARK_SIZE} ONNX predictions, got {len(onnx_predictions_df)}")
    if not np.isfinite(onnx_predictions_df["predicted_top1_probability"].to_numpy(dtype=np.float64)).all():
        raise ValueError("Non-finite probability detected in ONNX prediction CSV")
    onnx_predictions_df.to_csv(ONNX_PREDICTIONS_PATH, index=False, encoding="utf-8")

    onnx_top1_accuracy = round(float(onnx_predictions_df["top1_correct"].mean()), 6)
    onnx_top5_accuracy = round(float(onnx_predictions_df["top5_correct"].mean()), 6)
    onnx_top1_agreement = round(float((onnx_predictions_df["predicted_top1_index"].to_numpy() == predictions_df["predicted_top1_index"].to_numpy()).mean()), 6)
    onnx_speedup_vs_baseline = round(baseline_mean_latency_ms / float(onnx_model_only_latency_summary["mean_latency_ms"]), 6)
    onnx_size_reduction_pct = round((1.0 - (onnx_model_size_mb / baseline_model_size_mb)) * 100.0, 6)
    onnx_accuracy_delta = round(onnx_top1_accuracy - float(baseline_row["top1_accuracy"]), 6)

    onnx_row: dict[str, Any] = {column: None for column in RESULT_COLUMNS}
    onnx_row.update({
        "run_id": "milestone_3_onnx_fp32",
        "model_id": "efficientnetb0_fp32_onnx",
        "runtime": "ONNX Runtime",
        "precision": "FP32",
        "device": "CPU",
        "provider": "CPUExecutionProvider",
        "batch_size": BATCH_SIZE,
        "thread_count": THREAD_COUNT,
        "num_images": BENCHMARK_SIZE,
        "warmup_runs": WARMUP_RUNS,
        "model_size_mb": onnx_model_size_mb,
        "load_time_s": round(float(onnx_session_load_time_s), 6),
        "total_model_only_time_s": round(float(sum(onnx_model_only_latencies_ms) / 1000.0), 6),
        "total_end_to_end_time_s": round(float(sum(onnx_end_to_end_latencies_ms) / 1000.0), 6),
        "mean_latency_ms": round(float(onnx_model_only_latency_summary["mean_latency_ms"]), 6),
        "median_latency_ms": round(float(onnx_model_only_latency_summary["median_latency_ms"]), 6),
        "p95_latency_ms": round(float(onnx_model_only_latency_summary["p95_latency_ms"]), 6),
        "fps_model_only": round(float(onnx_model_only_latency_summary["fps"]), 6),
        "fps_end_to_end": round(float(onnx_end_to_end_summary["latency_summary"]["fps"]), 6),
        "top1_accuracy": onnx_top1_accuracy,
        "top5_accuracy": onnx_top5_accuracy,
        "top1_agreement": onnx_top1_agreement,
        "max_abs_output_diff": round(subset_max_abs_diff, 8),
        "mean_abs_output_diff": round(subset_mean_abs_diff, 8),
        "ram_avg_mb": onnx_model_only_resource_summary["ram_avg_mb"],
        "ram_peak_mb": onnx_model_only_resource_summary["ram_peak_mb"],
        "cpu_process_avg_pct": onnx_model_only_resource_summary["cpu_process_avg_pct"],
        "cpu_process_peak_pct": onnx_model_only_resource_summary["cpu_process_peak_pct"],
        "cpu_system_avg_pct": onnx_model_only_resource_summary["cpu_system_avg_pct"],
        "cpu_system_peak_pct": onnx_model_only_resource_summary["cpu_system_peak_pct"],
        "gpu_avg_pct": onnx_model_only_resource_summary["gpu_avg_pct"],
        "gpu_peak_pct": onnx_model_only_resource_summary["gpu_peak_pct"],
        "vram_avg_mb": onnx_model_only_resource_summary["vram_avg_mb"],
        "vram_peak_mb": onnx_model_only_resource_summary["vram_peak_mb"],
        "speedup_vs_baseline": onnx_speedup_vs_baseline,
        "size_reduction_pct": onnx_size_reduction_pct,
        "accuracy_delta": onnx_accuracy_delta,
        "notes": json.dumps({
            "onnx_opset": ONNX_OPSET,
            "subset_max_abs_diff": subset_max_abs_diff,
            "subset_mean_abs_diff": subset_mean_abs_diff,
            "subset_top1_agreement": subset_top1_agreement,
            "subset_top5_agreement": subset_top5_agreement,
            "onnx_provider": onnx_provider,
            "convert_time_s": round(float(onnx_convert_time_s), 6),
            "parse_time_s": round(float(onnx_parse_time_s), 6),
            "session_load_time_s": round(float(onnx_session_load_time_s), 6),
        }, ensure_ascii=False),
    })

    benchmark_df = pd.read_csv(BASELINE_BENCHMARK_PATH)
    benchmark_df = benchmark_df[benchmark_df["model_id"] != onnx_row["model_id"]] if "model_id" in benchmark_df.columns else benchmark_df
    benchmark_df = benchmark_df[benchmark_df["model_id"] != BASELINE_MODEL_ID] if "model_id" in benchmark_df.columns else benchmark_df
    updated_benchmark_df = pd.concat([benchmark_df, pd.DataFrame([baseline_row]), pd.DataFrame([onnx_row])], ignore_index=True)
    updated_benchmark_df = updated_benchmark_df.reindex(columns=RESULT_COLUMNS)
    updated_benchmark_df.to_csv(BASELINE_BENCHMARK_PATH, index=False, encoding="utf-8")

    onnx_validation = {
        "status": "PASS",
        "opset": ONNX_OPSET,
        "provider": onnx_provider,
        "checker": "PASS",
        "subset_max_abs_diff": subset_max_abs_diff,
        "subset_mean_abs_diff": subset_mean_abs_diff,
        "subset_top1_agreement": subset_top1_agreement,
        "subset_top5_agreement": subset_top5_agreement,
        "num_images": BENCHMARK_SIZE,
        "top1_accuracy": onnx_top1_accuracy,
        "top5_accuracy": onnx_top5_accuracy,
        "model_only_latency": onnx_model_only_latency_summary,
        "end_to_end_latency": onnx_end_to_end_summary["latency_summary"],
        "resource_metrics": onnx_model_only_resource_summary,
        "model_path": str(ONNX_MODEL_PATH),
        "predictions_path": str(ONNX_PREDICTIONS_PATH),
        "benchmark_rows": int(len(updated_benchmark_df)),
        "benchmark_table": updated_benchmark_df[["model_id", "runtime", "precision", "provider", "mean_latency_ms", "fps_model_only", "top1_accuracy", "top1_agreement", "size_reduction_pct", "speedup_vs_baseline"]].to_dict("records"),
        "speedup_vs_baseline": onnx_speedup_vs_baseline,
        "size_reduction_pct": onnx_size_reduction_pct,
        "accuracy_delta": onnx_accuracy_delta,
    }
    print("ONNX prediction sample:")
    print(onnx_predictions_df.head(5).to_string(index=False))
    print("Benchmark comparison TF vs ONNX:")
    comparison_df = pd.DataFrame([baseline_row, onnx_row])[["model_id", "runtime", "precision", "provider", "model_size_mb", "mean_latency_ms", "fps_model_only", "top1_accuracy", "top1_agreement", "size_reduction_pct", "speedup_vs_baseline", "accuracy_delta"]]
    print(comparison_df.to_string(index=False))
    print(json.dumps(onnx_validation, indent=2, ensure_ascii=False, default=str))



ONNX export start

ONNX model:

D:\DAT301m\slot17\models\onnx\efficientnetb0_fp32.onnx

ONNX size MB:

20.17

...

ONNX prediction sample:

sample_id                                               relative_path  ground_truth_index  predicted_top1_index predicted_top1_class  predicted_top1_probability            top5_indices                                                   top5_classes                                                                                               top5_probabilities  top1_correct  top5_correct
    b0000  data/raw/imagenette2-320/val/n01440764/n01440764_1310.JPEG                   0                     0                tench                    0.704822   [0, 1, 395, 392, 389]      ["tench", "goldfish", "gar", "rock_beauty", "barracouta"]       [0.7048221230506897, 0.03805689141154289, 0.032529670745134354, 0.02317734621465206, 0.015042091719806194]          True          True
    b0001 data/raw/imagenette2-320/val/n01440764/n01440764_14190.JPEG                   0                   389           barracouta                    0.742603 [389, 395, 394, 390, 0]              ["barracouta", "gar", "s

Benchmark comparison TF vs ONNX:

                    model_id          runtime precision             provider  model_size_mb  mean_latency_ms  fps_model_only  top1_accuracy  top1_agreement  size_reduction_pct  speedup_vs_baseline  accuracy_delta
efficientnetb0_fp32_baseline TensorFlow/Keras      FP32     TensorFlow/Keras          21.04       140.001183        7.142797          0.836             1.0            0.000000             1.000000             0.0
    efficientnetb0_fp32_onnx     ONNX Runtime      FP32 CPUExecutionProvider          20.17         9.016135      110.912265          0.836             1.0            4.134981            15.527848             0.0

{
  "status": "PASS",
  "opset": 13,
  "provider": [
    "CPUExecutionProvider"
  ],
  "checker": "PASS",
  "subset_max_abs_diff": 1.3709068298339844e-06,
  "subset_mean_abs_diff": 1.0633635555423738e-09,
  "subset_top1_agreement": 1.0,
  "subset_top5_agreement": 1.0,
  "num_images": 500,
  "top1_accuracy": 0.836,
  "top5_accuracy": 0.974,
  "model_only_latency": {
    "count": 500.0,
    "mean_latency_ms": 9.0161353999938,
    "median_latency_ms": 8.96535000174481,
    "p95_latency_ms": 10.34518499927799,
    "fps": 110.91226513753195
  },
  "end_to_end_latency": {
    "count": 500.0,
    "mean_latency_ms": 12.86308239997743,
    "median_latency_ms": 12.561200001073303,
    "p95_latency_ms": 15.901780002241138,
    "fps": 77.74186380099336
  },
  "resource_metrics": {
    "count": 502,
    "ram_avg_mb": 1256.83,
    "ram_peak_mb": 1256.83,
    "cpu_process_avg_pct": 657.16,
    "cpu_process_peak_pct": 1562.5,
    "cpu_system_avg_pct": 49.96,
    "cpu_system_peak_pct": 100.0,
    "gpu_

In [13]:
final_status = "READY_FOR_MILESTONE_4" if 'onnx_validation' in globals() and onnx_validation.get('status') == 'PASS' and Path(ONNX_MODEL_PATH).exists() and Path(ONNX_PREDICTIONS_PATH).exists() else (onnx_validation.get('status') if 'onnx_validation' in globals() else 'BLOCKED_ON_ONNX')
milestone3_report = {
    "status": final_status,
    "onnx_path": str(ONNX_MODEL_PATH),
    "onnx_size_mb": round(ONNX_MODEL_PATH.stat().st_size / (1024 ** 2), 2) if Path(ONNX_MODEL_PATH).exists() else None,
    "opset": ONNX_OPSET,
    "provider": onnx_validation.get("provider") if 'onnx_validation' in globals() else None,
    "checker": onnx_validation.get("checker") if 'onnx_validation' in globals() else None,
    "numerical_equivalence": {
        "subset_max_abs_diff": onnx_validation.get("subset_max_abs_diff") if 'onnx_validation' in globals() else None,
        "subset_mean_abs_diff": onnx_validation.get("subset_mean_abs_diff") if 'onnx_validation' in globals() else None,
        "subset_top1_agreement": onnx_validation.get("subset_top1_agreement") if 'onnx_validation' in globals() else None,
        "subset_top5_agreement": onnx_validation.get("subset_top5_agreement") if 'onnx_validation' in globals() else None,
    },
    "num_images": onnx_validation.get("num_images") if 'onnx_validation' in globals() else None,
    "top1_accuracy": onnx_validation.get("top1_accuracy") if 'onnx_validation' in globals() else None,
    "top5_accuracy": onnx_validation.get("top5_accuracy") if 'onnx_validation' in globals() else None,
    "model_only_latency": onnx_validation.get("model_only_latency") if 'onnx_validation' in globals() else None,
    "end_to_end_latency": onnx_validation.get("end_to_end_latency") if 'onnx_validation' in globals() else None,
    "resource_metrics": onnx_validation.get("resource_metrics") if 'onnx_validation' in globals() else None,
    "speedup_vs_baseline": onnx_validation.get("speedup_vs_baseline") if 'onnx_validation' in globals() else None,
    "size_reduction_pct": onnx_validation.get("size_reduction_pct") if 'onnx_validation' in globals() else None,
    "prediction_file": str(ONNX_PREDICTIONS_PATH),
    "benchmark_file": str(BASELINE_BENCHMARK_PATH),
    "notebook_output_status": "SAVED_INPLACE",
    "blockers": [] if final_status == "READY_FOR_MILESTONE_4" else [final_status],
    "next_milestone": "Milestone 4" if final_status == "READY_FOR_MILESTONE_4" else "Milestone 3 follow-up",
}
print(json.dumps(milestone3_report, indent=2, ensure_ascii=False, default=str))


{
  "status": "READY_FOR_MILESTONE_4",
  "onnx_path": "D:\\DAT301m\\slot17\\models\\onnx\\efficientnetb0_fp32.onnx",
  "onnx_size_mb": 20.17,
  "opset": 13,
  "provider": [
    "CPUExecutionProvider"
  ],
  "checker": "PASS",
  "numerical_equivalence": {
    "subset_max_abs_diff": 1.3709068298339844e-06,
    "subset_mean_abs_diff": 1.0633635555423738e-09,
    "subset_top1_agreement": 1.0,
    "subset_top5_agreement": 1.0
  },
  "num_images": 500,
  "top1_accuracy": 0.836,
  "top5_accuracy": 0.974,
  "model_only_latency": {
    "count": 500.0,
    "mean_latency_ms": 9.0161353999938,
    "median_latency_ms": 8.96535000174481,
    "p95_latency_ms": 10.34518499927799,
    "fps": 110.91226513753195
  },
  "end_to_end_latency": {
    "count": 500.0,
    "mean_latency_ms": 12.86308239997743,
    "median_latency_ms": 12.561200001073303,
    "p95_latency_ms": 15.901780002241138,
    "fps": 77.74186380099336
  },
  "resource_metrics": {
    "count": 502,
    "ram_avg_mb": 1256.83,
    "ram_peak_

## Milestone 4: TFLite FP32

Ph?n n?y chuy?n baseline TensorFlow/Keras sang TFLite FP32, x?c minh `tf.lite.Interpreter`, ki?m tra t??ng ???ng s? h?c tr?n 10 ?nh, r?i benchmark 500 ?nh tr?n CPU. `CPUExecutionProvider` c?a ONNX kh?ng li?n quan ? milestone n?y. Vi?c ?o `psutil` CPU process c? th? v??t 100% tr?n m?y nhi?u logical processors l? b?nh th??ng v? gi? tr? n?y ???c c?ng tr?n nhi?u l?i.


In [14]:
TFLITE_MODEL_DIR = PROJECT_ROOT / "models" / "tflite"
TFLITE_MODEL_DIR.mkdir(parents=True, exist_ok=True)
TFLITE_MODEL_PATH = TFLITE_MODEL_DIR / "efficientnetb0_fp32.tflite"
TFLITE_PREDICTIONS_PATH = RESULTS_DIR / "predictions_tflite.csv"
TFLITE_SUBSET_SIZE = min(10, BENCHMARK_SIZE)
TFLITE_THREADS = THREAD_COUNT

baseline_results_df = pd.read_csv(BASELINE_BENCHMARK_PATH)
baseline_row = baseline_results_df.loc[baseline_results_df["model_id"] == BASELINE_MODEL_ID].iloc[0].to_dict()
onnx_row = baseline_results_df.loc[baseline_results_df["model_id"] == "efficientnetb0_fp32_onnx"].iloc[0].to_dict()
baseline_model_size_mb = float(baseline_row["model_size_mb"])
baseline_mean_latency_ms = float(baseline_row["mean_latency_ms"])
onnx_mean_latency_ms = float(onnx_row["mean_latency_ms"])

print("TFLite conversion start")
tflite_convert_start = time.perf_counter()
tflite_converter = tf.lite.TFLiteConverter.from_keras_model(tf_model)
tflite_converter.optimizations = []
tflite_model_bytes = tflite_converter.convert()
tflite_convert_time_s = time.perf_counter() - tflite_convert_start
TFLITE_MODEL_PATH.write_bytes(tflite_model_bytes)
tflite_model_size_mb = round(TFLITE_MODEL_PATH.stat().st_size / (1024 ** 2), 2)

interpreter_load_start = time.perf_counter()
tflite_interpreter = tf.lite.Interpreter(model_path=str(TFLITE_MODEL_PATH), num_threads=TFLITE_THREADS)
tflite_interpreter.allocate_tensors()
tflite_load_time_s = time.perf_counter() - interpreter_load_start

tflite_input_details_raw = tflite_interpreter.get_input_details()[0]
tflite_output_details_raw = tflite_interpreter.get_output_details()[0]
tflite_input_details = {
    "name": tflite_input_details_raw["name"],
    "shape": tflite_input_details_raw["shape"].tolist(),
    "dtype": str(tflite_input_details_raw["dtype"].__name__),
    "index": int(tflite_input_details_raw["index"]),
}
tflite_output_details = {
    "name": tflite_output_details_raw["name"],
    "shape": tflite_output_details_raw["shape"].tolist(),
    "dtype": str(tflite_output_details_raw["dtype"].__name__),
    "index": int(tflite_output_details_raw["index"]),
}
print("TFLite model:", TFLITE_MODEL_PATH)
print("TFLite size MB:", tflite_model_size_mb)
print("TFLite load time s:", round(float(tflite_load_time_s), 6))
print("TFLite input details:", json.dumps(tflite_input_details, indent=2, ensure_ascii=False))
print("TFLite output details:", json.dumps(tflite_output_details, indent=2, ensure_ascii=False))
print("TFLite interpreter threads:", TFLITE_THREADS)

if tflite_input_details["shape"] != [1, 224, 224, 3] or tflite_output_details["shape"] != [1, 1000]:
    raise ValueError(f"Unexpected TFLite tensor shapes: input={tflite_input_details['shape']}, output={tflite_output_details['shape']}")
if tflite_input_details["dtype"] != "float32" or tflite_output_details["dtype"] != "float32":
    raise ValueError(f"Unexpected TFLite tensor dtypes: input={tflite_input_details['dtype']}, output={tflite_output_details['dtype']}")

subset_records = manifest_records[:TFLITE_SUBSET_SIZE]
subset_tensors = preloaded_tensors[:TFLITE_SUBSET_SIZE]
tf_subset_outputs = []
tflite_subset_outputs = []
for tensor in subset_tensors:
    tf_subset_outputs.append(tf_model(tf.convert_to_tensor(tensor[None, ...], dtype=tf.float32), training=False).numpy()[0])
    tflite_interpreter.set_tensor(tflite_input_details["index"], tensor[None, ...].astype(np.float32, copy=False))
    tflite_interpreter.invoke()
    tflite_subset_outputs.append(tflite_interpreter.get_tensor(tflite_output_details["index"])[0])

subset_rows = []
for record, tf_probs, tflite_probs in zip(subset_records, tf_subset_outputs, tflite_subset_outputs):
    tf_top5 = decode_topk(tf_probs, top_k=5)
    tflite_top5 = decode_topk(tflite_probs, top_k=5)
    subset_rows.append({
        "sample_id": record["sample_id"],
        "max_abs_diff": float(np.max(np.abs(tf_probs - tflite_probs))),
        "mean_abs_diff": float(np.mean(np.abs(tf_probs - tflite_probs))),
        "top1_agree": bool(tf_top5["indices"][0] == tflite_top5["indices"][0]),
        "top5_agree": bool(tf_top5["indices"] == tflite_top5["indices"]),
    })
subset_df = pd.DataFrame(subset_rows)
subset_max_abs_diff = float(subset_df["max_abs_diff"].max())
subset_mean_abs_diff = float(subset_df["mean_abs_diff"].mean())
subset_top1_agreement = float(subset_df["top1_agree"].mean())
subset_top5_agreement = float(subset_df["top5_agree"].mean())
print("Numerical equivalence table:")
print(subset_df.to_string(index=False))

if subset_max_abs_diff > 1e-3 or subset_top1_agreement < 0.9 or subset_top5_agreement < 0.9 or not np.isfinite(subset_df[["max_abs_diff", "mean_abs_diff"]].to_numpy()).all():
    tflite_validation = {
        "status": "BLOCKED_ON_TFLITE_VALIDATION",
        "reason": "output_diff_or_agreement_too_low",
        "subset_max_abs_diff": subset_max_abs_diff,
        "subset_mean_abs_diff": subset_mean_abs_diff,
        "subset_top1_agreement": subset_top1_agreement,
        "subset_top5_agreement": subset_top5_agreement,
        "convert_time_s": round(float(tflite_convert_time_s), 6),
        "load_time_s": round(float(tflite_load_time_s), 6),
        "input_details": tflite_input_details,
        "output_details": tflite_output_details,
        "model_path": str(TFLITE_MODEL_PATH),
    }
    print(json.dumps(tflite_validation, indent=2, ensure_ascii=False))
else:
    print("TFLite numerical validation: PASS")

    def run_tflite_model_only(records: list[dict[str, Any]], tensors: list[np.ndarray]) -> tuple[list[dict[str, Any]], dict[str, Any], dict[str, Any], list[float]]:
        monitor = ResourceMonitor()
        monitor.process.cpu_percent(None)
        psutil.cpu_percent(None)
        predictions: list[dict[str, Any]] = []
        latencies_ms: list[float] = []
        monitor.sample(notes="tflite_model_only_start")
        for record, tensor in zip(records, tensors):
            start = time.perf_counter()
            tflite_interpreter.set_tensor(tflite_input_details["index"], tensor[None, ...].astype(np.float32, copy=False))
            tflite_interpreter.invoke()
            probabilities = tflite_interpreter.get_tensor(tflite_output_details["index"])[0]
            elapsed_ms = (time.perf_counter() - start) * 1000.0
            if not np.isfinite(probabilities).all():
                raise ValueError(f"Non-finite TFLite output for {record['relative_path']}")
            decoded = decode_topk(probabilities, top_k=5)
            top1_index = decoded["indices"][0]
            top5_indices = decoded["indices"]
            ground_truth_index = int(record["ground_truth_index"])
            predictions.append({
                "sample_id": record["sample_id"],
                "relative_path": record["relative_path"],
                "ground_truth_index": ground_truth_index,
                "predicted_top1_index": top1_index,
                "predicted_top1_class": decoded["class_names"][0],
                "predicted_top1_probability": float(decoded["probabilities"][0]),
                "top5_indices": json.dumps(top5_indices, ensure_ascii=False),
                "top5_classes": json.dumps(decoded["class_names"], ensure_ascii=False),
                "top5_probabilities": json.dumps([float(value) for value in decoded["probabilities"]], ensure_ascii=False),
                "top1_correct": bool(top1_index == ground_truth_index),
                "top5_correct": bool(ground_truth_index in top5_indices),
            })
            latencies_ms.append(elapsed_ms)
            monitor.sample(notes="tflite_model_only")
        monitor.sample(notes="tflite_model_only_end")
        return predictions, summarize_latencies(latencies_ms), summarize_resource_samples(monitor.samples), latencies_ms

    def run_tflite_end_to_end(records: list[dict[str, Any]]) -> tuple[dict[str, Any], list[float]]:
        monitor = ResourceMonitor()
        monitor.process.cpu_percent(None)
        psutil.cpu_percent(None)
        latencies_ms: list[float] = []
        monitor.sample(notes="tflite_end_to_end_start")
        for record in records:
            start = time.perf_counter()
            tensor = prepare_image_tensor(Path(record.get("absolute_path", PROJECT_ROOT / record["relative_path"])))
            tflite_interpreter.set_tensor(tflite_input_details["index"], tensor[None, ...].astype(np.float32, copy=False))
            tflite_interpreter.invoke()
            probabilities = tflite_interpreter.get_tensor(tflite_output_details["index"])[0]
            elapsed_ms = (time.perf_counter() - start) * 1000.0
            if not np.isfinite(probabilities).all():
                raise ValueError(f"Non-finite TFLite output for {record['relative_path']}")
            latencies_ms.append(elapsed_ms)
            monitor.sample(notes="tflite_end_to_end")
        monitor.sample(notes="tflite_end_to_end_end")
        return {"latency_summary": summarize_latencies(latencies_ms), "resource_summary": summarize_resource_samples(monitor.samples)}, latencies_ms

    tflite_model_only_predictions, tflite_model_only_latency_summary, tflite_model_only_resource_summary, tflite_model_only_latencies_ms = run_tflite_model_only(manifest_records, preloaded_tensors)
    tflite_end_to_end_summary, tflite_end_to_end_latencies_ms = run_tflite_end_to_end(manifest_records)
    tflite_predictions_df = pd.DataFrame(tflite_model_only_predictions)
    if len(tflite_predictions_df) != BENCHMARK_SIZE:
        raise ValueError(f"Expected {BENCHMARK_SIZE} TFLite predictions, got {len(tflite_predictions_df)}")
    if not np.isfinite(tflite_predictions_df["predicted_top1_probability"].to_numpy(dtype=np.float64)).all():
        raise ValueError("Non-finite probability detected in TFLite prediction CSV")
    tflite_predictions_df.to_csv(TFLITE_PREDICTIONS_PATH, index=False, encoding="utf-8")

    tflite_top1_accuracy = round(float(tflite_predictions_df["top1_correct"].mean()), 6)
    tflite_top5_accuracy = round(float(tflite_predictions_df["top5_correct"].mean()), 6)
    tflite_top1_agreement = round(float((tflite_predictions_df["predicted_top1_index"].to_numpy() == predictions_df["predicted_top1_index"].to_numpy()).mean()), 6)
    tflite_speedup_vs_baseline = round(baseline_mean_latency_ms / float(tflite_model_only_latency_summary["mean_latency_ms"]), 6)
    tflite_vs_onnx_ratio = round(float(onnx_mean_latency_ms) / float(tflite_model_only_latency_summary["mean_latency_ms"]), 6)
    tflite_size_reduction_pct = round((1.0 - (tflite_model_size_mb / baseline_model_size_mb)) * 100.0, 6)
    tflite_accuracy_delta = round(tflite_top1_accuracy - float(baseline_row["top1_accuracy"]), 6)

    tflite_row: dict[str, Any] = {column: None for column in RESULT_COLUMNS}
    tflite_row.update({
        "run_id": "milestone_4_tflite_fp32",
        "model_id": "efficientnetb0_fp32_tflite",
        "runtime": "TFLite Interpreter",
        "precision": "FP32",
        "device": "CPU",
        "provider": "TFLite",
        "batch_size": BATCH_SIZE,
        "thread_count": TFLITE_THREADS,
        "num_images": BENCHMARK_SIZE,
        "warmup_runs": WARMUP_RUNS,
        "model_size_mb": tflite_model_size_mb,
        "load_time_s": round(float(tflite_load_time_s), 6),
        "total_model_only_time_s": round(float(sum(tflite_model_only_latencies_ms) / 1000.0), 6),
        "total_end_to_end_time_s": round(float(sum(tflite_end_to_end_latencies_ms) / 1000.0), 6),
        "mean_latency_ms": round(float(tflite_model_only_latency_summary["mean_latency_ms"]), 6),
        "median_latency_ms": round(float(tflite_model_only_latency_summary["median_latency_ms"]), 6),
        "p95_latency_ms": round(float(tflite_model_only_latency_summary["p95_latency_ms"]), 6),
        "fps_model_only": round(float(tflite_model_only_latency_summary["fps"]), 6),
        "fps_end_to_end": round(float(tflite_end_to_end_summary["latency_summary"]["fps"]), 6),
        "top1_accuracy": tflite_top1_accuracy,
        "top5_accuracy": tflite_top5_accuracy,
        "top1_agreement": tflite_top1_agreement,
        "max_abs_output_diff": round(subset_max_abs_diff, 8),
        "mean_abs_output_diff": round(subset_mean_abs_diff, 8),
        "ram_avg_mb": tflite_model_only_resource_summary["ram_avg_mb"],
        "ram_peak_mb": tflite_model_only_resource_summary["ram_peak_mb"],
        "cpu_process_avg_pct": tflite_model_only_resource_summary["cpu_process_avg_pct"],
        "cpu_process_peak_pct": tflite_model_only_resource_summary["cpu_process_peak_pct"],
        "cpu_system_avg_pct": tflite_model_only_resource_summary["cpu_system_avg_pct"],
        "cpu_system_peak_pct": tflite_model_only_resource_summary["cpu_system_peak_pct"],
        "gpu_avg_pct": tflite_model_only_resource_summary["gpu_avg_pct"],
        "gpu_peak_pct": tflite_model_only_resource_summary["gpu_peak_pct"],
        "vram_avg_mb": tflite_model_only_resource_summary["vram_avg_mb"],
        "vram_peak_mb": tflite_model_only_resource_summary["vram_peak_mb"],
        "speedup_vs_baseline": tflite_speedup_vs_baseline,
        "size_reduction_pct": tflite_size_reduction_pct,
        "accuracy_delta": tflite_accuracy_delta,
        "notes": json.dumps({
            "tflite_threads": TFLITE_THREADS,
            "input_details": tflite_input_details,
            "output_details": tflite_output_details,
            "subset_max_abs_diff": subset_max_abs_diff,
            "subset_mean_abs_diff": subset_mean_abs_diff,
            "subset_top1_agreement": subset_top1_agreement,
            "subset_top5_agreement": subset_top5_agreement,
            "onnx_comparison_mean_latency_ms": onnx_mean_latency_ms,
            "speedup_vs_onnx_fp32": tflite_vs_onnx_ratio,
            "speedup_or_slowdown_vs_onnx_fp32": "speedup" if tflite_vs_onnx_ratio >= 1.0 else "slowdown",
            "converter": "TensorFlow 2.15.1 TFLiteConverter",
        }, ensure_ascii=False),
    })

    updated_benchmark_df = pd.DataFrame([baseline_row, onnx_row, tflite_row]).reindex(columns=RESULT_COLUMNS)
    updated_benchmark_df.to_csv(BASELINE_BENCHMARK_PATH, index=False, encoding="utf-8")

    tflite_validation = {
        "status": "PASS",
        "converter": "TensorFlow 2.15.1 TFLiteConverter",
        "load_time_s": round(float(tflite_load_time_s), 6),
        "input_details": tflite_input_details,
        "output_details": tflite_output_details,
        "subset_max_abs_diff": subset_max_abs_diff,
        "subset_mean_abs_diff": subset_mean_abs_diff,
        "subset_top1_agreement": subset_top1_agreement,
        "subset_top5_agreement": subset_top5_agreement,
        "num_images": BENCHMARK_SIZE,
        "top1_accuracy": tflite_top1_accuracy,
        "top5_accuracy": tflite_top5_accuracy,
        "model_only_latency": tflite_model_only_latency_summary,
        "end_to_end_latency": tflite_end_to_end_summary["latency_summary"],
        "resource_metrics": tflite_model_only_resource_summary,
        "model_path": str(TFLITE_MODEL_PATH),
        "predictions_path": str(TFLITE_PREDICTIONS_PATH),
        "benchmark_rows": int(len(updated_benchmark_df)),
        "benchmark_table": updated_benchmark_df[["model_id", "runtime", "precision", "provider", "mean_latency_ms", "fps_model_only", "top1_accuracy", "top1_agreement", "size_reduction_pct", "speedup_vs_baseline"]].to_dict("records"),
        "speedup_vs_baseline": tflite_speedup_vs_baseline,
        "speedup_vs_onnx_fp32": tflite_vs_onnx_ratio,
        "size_reduction_pct": tflite_size_reduction_pct,
        "accuracy_delta": tflite_accuracy_delta,
    }
    print("TFLite prediction sample:")
    print(tflite_predictions_df.head(5).to_string(index=False))
    print("Benchmark comparison TF vs ONNX vs TFLite:")
    comparison_df = pd.DataFrame([baseline_row, onnx_row, tflite_row])[ ["model_id", "runtime", "precision", "provider", "model_size_mb", "mean_latency_ms", "fps_model_only", "top1_accuracy", "top1_agreement", "size_reduction_pct", "speedup_vs_baseline", "accuracy_delta"] ]
    print(comparison_df.to_string(index=False))
    print(json.dumps(tflite_validation, indent=2, ensure_ascii=False, default=str))



TFLite conversion start

INFO:tensorflow:Assets written to: C:\Users\klein\AppData\Local\Temp\tmpeexsmg8x\assets


INFO:tensorflow:Assets written to: C:\Users\klein\AppData\Local\Temp\tmpeexsmg8x\assets


TFLite model:

D:\DAT301m\slot17\models\tflite\efficientnetb0_fp32.tflite

...

TFLite prediction sample:

sample_id                                               relative_path  ground_truth_index  predicted_top1_index predicted_top1_class  predicted_top1_probability            top5_indices                                                   top5_classes                                                                                              top5_probabilities  top1_correct  top5_correct
    b0000  data/raw/imagenette2-320/val/n01440764/n01440764_1310.JPEG                   0                     0                tench                    0.704823   [0, 1, 395, 392, 389]      ["tench", "goldfish", "gar", "rock_beauty", "barracouta"]      [0.7048234343528748, 0.03805670887231827, 0.032529328018426895, 0.02317730151116848, 0.015041983686387539]          True          True
    b0001 data/raw/imagenette2-320/val/n01440764/n01440764_14190.JPEG                   0                   389           barracouta                    0.742603 [389, 395, 394, 390, 0]              ["barracouta", "gar", "stu

Benchmark comparison TF vs ONNX vs TFLite:

                    model_id            runtime precision             provider  model_size_mb  mean_latency_ms  fps_model_only  top1_accuracy  top1_agreement  size_reduction_pct  speedup_vs_baseline  accuracy_delta
efficientnetb0_fp32_baseline   TensorFlow/Keras      FP32     TensorFlow/Keras          21.04       140.001183        7.142797          0.836             1.0            0.000000             1.000000             0.0
    efficientnetb0_fp32_onnx       ONNX Runtime      FP32 CPUExecutionProvider          20.17         9.016135      110.912265          0.836             1.0            4.134981            15.527848             0.0
  efficientnetb0_fp32_tflite TFLite Interpreter      FP32               TFLite          20.18        54.782744       18.253923          0.836             1.0            4.087452             2.555571             0.0

{
  "status": "PASS",
  "converter": "TensorFlow 2.15.1 TFLiteConverter",
  "load_time_s": 0.00965,
  "input_details": {
    "name": "serving_default_input_1:0",
    "shape": [
      1,
      224,
      224,
      3
    ],
    "dtype": "float32",
    "index": 0
  },
  "output_details": {
    "name": "StatefulPartitionedCall:0",
    "shape": [
      1,
      1000
    ],
    "dtype": "float32",
    "index": 493
  },
  "subset_max_abs_diff": 1.6689300537109375e-06,
  "subset_mean_abs_diff": 1.4003690623765054e-09,
  "subset_top1_agreement": 1.0,
  "subset_top5_agreement": 1.0,
  "num_images": 500,
  "top1_accuracy": 0.836,
  "top5_accuracy": 0.974,
  "model_only_latency": {
    "count": 500.0,
    "mean_latency_ms": 54.78274420012167,
    "median_latency_ms": 53.84725000112667,
    "p95_latency_ms": 61.398064998866175,
    "fps": 18.25392310299379
  },
  "end_to_end_latency": {
    "count": 500.0,
    "mean_latency_ms": 56.47466000001441,
    "median_latency_ms": 55.92814999727125,
    "p

In [15]:
final_status = "READY_FOR_MILESTONE_5" if 'tflite_validation' in globals() and tflite_validation.get('status') == 'PASS' and Path(TFLITE_MODEL_PATH).exists() and Path(TFLITE_PREDICTIONS_PATH).exists() else (tflite_validation.get('status') if 'tflite_validation' in globals() else 'BLOCKED_ON_TFLITE')
milestone4_report = {
    "status": final_status,
    "tflite_path": str(TFLITE_MODEL_PATH),
    "tflite_size_mb": round(TFLITE_MODEL_PATH.stat().st_size / (1024 ** 2), 2) if Path(TFLITE_MODEL_PATH).exists() else None,
    "converter": tflite_validation.get("converter") if 'tflite_validation' in globals() else None,
    "input_details": tflite_validation.get("input_details") if 'tflite_validation' in globals() else None,
    "output_details": tflite_validation.get("output_details") if 'tflite_validation' in globals() else None,
    "numerical_equivalence": {
        "subset_max_abs_diff": tflite_validation.get("subset_max_abs_diff") if 'tflite_validation' in globals() else None,
        "subset_mean_abs_diff": tflite_validation.get("subset_mean_abs_diff") if 'tflite_validation' in globals() else None,
        "subset_top1_agreement": tflite_validation.get("subset_top1_agreement") if 'tflite_validation' in globals() else None,
        "subset_top5_agreement": tflite_validation.get("subset_top5_agreement") if 'tflite_validation' in globals() else None,
    },
    "num_images": tflite_validation.get("num_images") if 'tflite_validation' in globals() else None,
    "top1_accuracy": tflite_validation.get("top1_accuracy") if 'tflite_validation' in globals() else None,
    "top5_accuracy": tflite_validation.get("top5_accuracy") if 'tflite_validation' in globals() else None,
    "model_only_latency": tflite_validation.get("model_only_latency") if 'tflite_validation' in globals() else None,
    "end_to_end_latency": tflite_validation.get("end_to_end_latency") if 'tflite_validation' in globals() else None,
    "resource_metrics": tflite_validation.get("resource_metrics") if 'tflite_validation' in globals() else None,
    "speedup_vs_baseline": tflite_validation.get("speedup_vs_baseline") if 'tflite_validation' in globals() else None,
    "speedup_vs_onnx_fp32": tflite_validation.get("speedup_vs_onnx_fp32") if 'tflite_validation' in globals() else None,
    "size_reduction_pct": tflite_validation.get("size_reduction_pct") if 'tflite_validation' in globals() else None,
    "prediction_file": str(TFLITE_PREDICTIONS_PATH),
    "benchmark_file": str(BASELINE_BENCHMARK_PATH),
    "notebook_output_status": "SAVED_INPLACE",
    "blockers": [] if final_status == "READY_FOR_MILESTONE_5" else [final_status],
    "next_milestone": "Milestone 5" if final_status == "READY_FOR_MILESTONE_5" else "Milestone 4 follow-up",
}
print(json.dumps(milestone4_report, indent=2, ensure_ascii=False, default=str))


{
  "status": "READY_FOR_MILESTONE_5",
  "tflite_path": "D:\\DAT301m\\slot17\\models\\tflite\\efficientnetb0_fp32.tflite",
  "tflite_size_mb": 20.18,
  "converter": "TensorFlow 2.15.1 TFLiteConverter",
  "input_details": {
    "name": "serving_default_input_1:0",
    "shape": [
      1,
      224,
      224,
      3
    ],
    "dtype": "float32",
    "index": 0
  },
  "output_details": {
    "name": "StatefulPartitionedCall:0",
    "shape": [
      1,
      1000
    ],
    "dtype": "float32",
    "index": 493
  },
  "numerical_equivalence": {
    "subset_max_abs_diff": 1.6689300537109375e-06,
    "subset_mean_abs_diff": 1.4003690623765054e-09,
    "subset_top1_agreement": 1.0,
    "subset_top5_agreement": 1.0
  },
  "num_images": 500,
  "top1_accuracy": 0.836,
  "top5_accuracy": 0.974,
  "model_only_latency": {
    "count": 500.0,
    "mean_latency_ms": 54.78274420012167,
    "median_latency_ms": 53.84725000112667,
    "p95_latency_ms": 61.398064998866175,
    "fps": 18.25392310299379



## Milestone 5: Post-Training Static INT8 Quantization

Ph?n n?y t?o hai bi?n th? INT8 tr?n CPU:

- ONNX Runtime INT8 v?i calibration t? `calibration_100.csv`
- TFLite INT8 full integer v?i representative dataset t? c?ng calibration set

T?t c? ph?p ?o v?n d?ng ??ng 500 ?nh trong `sample_500.csv`, batch size = 1, kh?ng thay ??i k?t qu? FP32 ?? c?.


In [16]:

import tempfile
from onnx import TensorProto, helper
from onnxruntime.quantization import CalibrationDataReader, CalibrationMethod, QuantFormat, QuantType, quantize_static

ONNX_INT8_MODEL_ID = "efficientnetb0_int8_onnx"
TFLITE_INT8_MODEL_ID = "efficientnetb0_int8_tflite"
ONNX_INT8_MODEL_PATH = PROJECT_ROOT / "models" / "onnx" / "efficientnetb0_int8.onnx"
TFLITE_INT8_MODEL_PATH = PROJECT_ROOT / "models" / "tflite" / "efficientnetb0_int8.tflite"
ONNX_INT8_PREDICTIONS_PATH = RESULTS_DIR / "predictions_onnx_int8.csv"
TFLITE_INT8_PREDICTIONS_PATH = RESULTS_DIR / "predictions_tflite_int8.csv"
calibration_records = pd.read_csv(DATASET_PATHS["calibration_manifest"]).assign(absolute_path=lambda df: df["relative_path"].map(lambda x: str(PROJECT_ROOT / x))).to_dict("records")
baseline_results_df = pd.read_csv(BASELINE_BENCHMARK_PATH)
baseline_row = baseline_results_df.loc[baseline_results_df["model_id"] == BASELINE_MODEL_ID].iloc[0].to_dict()
onnx_fp32_row = baseline_results_df.loc[baseline_results_df["model_id"] == "efficientnetb0_fp32_onnx"].iloc[0].to_dict()
tflite_fp32_row = baseline_results_df.loc[baseline_results_df["model_id"] == "efficientnetb0_fp32_tflite"].iloc[0].to_dict()
baseline_model_size_mb = float(baseline_row["model_size_mb"])
baseline_top1_accuracy = float(baseline_row["top1_accuracy"])
baseline_mean_latency_ms = float(baseline_row["mean_latency_ms"])
onnx_fp32_mean_latency_ms = float(onnx_fp32_row["mean_latency_ms"])
tflite_fp32_mean_latency_ms = float(tflite_fp32_row["mean_latency_ms"])
onnx_fp32_model_size_mb = float(onnx_fp32_row["model_size_mb"])
tflite_fp32_model_size_mb = float(tflite_fp32_row["model_size_mb"])

def softmax_np(logits):
    logits = np.asarray(logits, dtype=np.float32)
    logits = logits - np.max(logits, axis=-1, keepdims=True)
    exp = np.exp(logits)
    return exp / np.sum(exp, axis=-1, keepdims=True)

def probability_to_prediction_row(record, probabilities):
    if not np.isfinite(probabilities).all():
        raise ValueError(f"Non-finite output detected for {record['relative_path']}")
    decoded = decode_topk(probabilities, top_k=5)
    top1_index = int(decoded["indices"][0])
    top5_indices = [int(v) for v in decoded["indices"]]
    gt = int(record["ground_truth_index"])
    return {
        "sample_id": record["sample_id"],
        "relative_path": record["relative_path"],
        "ground_truth_index": gt,
        "predicted_top1_index": top1_index,
        "predicted_top1_class": decoded["class_names"][0],
        "predicted_top1_probability": float(decoded["probabilities"][0]),
        "top5_indices": json.dumps(top5_indices, ensure_ascii=False),
        "top5_classes": json.dumps(decoded["class_names"], ensure_ascii=False),
        "top5_probabilities": json.dumps([float(v) for v in decoded["probabilities"]], ensure_ascii=False),
        "top1_correct": bool(top1_index == gt),
        "top5_correct": bool(gt in top5_indices),
    }

def quantize_tflite_tensor(tensor, input_details):
    scale = float(input_details["scale"])
    zero_point = int(input_details["zero_point"])
    if scale == 0:
        raise ValueError("TFLite input quantization scale must be non-zero")
    info = np.iinfo(input_details["dtype"])
    q = np.round(tensor / scale + zero_point)
    return np.clip(q, info.min, info.max).astype(input_details["dtype"])

def dequantize_tflite_tensor(tensor, output_details):
    scale = float(output_details["scale"])
    zero_point = int(output_details["zero_point"])
    return (tensor.astype(np.float32) - zero_point) * scale

def save_prediction_csv(rows, path):
    df = pd.DataFrame(rows)
    if len(df) != BENCHMARK_SIZE:
        raise ValueError(f"Expected {BENCHMARK_SIZE} rows, got {len(df)}")
    if not np.isfinite(df["predicted_top1_probability"].to_numpy(dtype=np.float64)).all():
        raise ValueError(f"Non-finite prediction probabilities detected in {path.name}")
    df.to_csv(path, index=False, encoding="utf-8")
    return df

def update_benchmark_results(new_rows):
    existing_df = pd.read_csv(BASELINE_BENCHMARK_PATH)
    existing_df = existing_df[~existing_df["model_id"].isin({ONNX_INT8_MODEL_ID, TFLITE_INT8_MODEL_ID})].copy()
    updated_df = pd.concat([existing_df, pd.DataFrame(new_rows)], ignore_index=True)
    updated_df = updated_df.reindex(columns=RESULT_COLUMNS)
    updated_df.to_csv(BASELINE_BENCHMARK_PATH, index=False, encoding="utf-8")
    return updated_df

print(json.dumps({
    "onnx_int8_model_path": str(ONNX_INT8_MODEL_PATH),
    "tflite_int8_model_path": str(TFLITE_INT8_MODEL_PATH),
    "calibration_rows": len(calibration_records),
    "benchmark_rows": len(manifest_records),
    "onnx_configuration": {"quantization_format": "QDQ", "activation_type": "QUInt8", "weight_type": "QInt8", "calibration_method": "MinMax", "operators_quantized": ["Conv", "MatMul", "Gemm"]},
    "tflite_configuration": {"supported_ops": ["TFLITE_BUILTINS_INT8"], "inference_input_type": "int8", "inference_output_type": "int8", "representative_dataset": "calibration_100.csv"}
}, indent=2, ensure_ascii=False))


{
  "onnx_int8_model_path": "D:\\DAT301m\\slot17\\models\\onnx\\efficientnetb0_int8.onnx",
  "tflite_int8_model_path": "D:\\DAT301m\\slot17\\models\\tflite\\efficientnetb0_int8.tflite",
  "calibration_rows": 100,
  "benchmark_rows": 500,
  "onnx_configuration": {
    "quantization_format": "QDQ",
    "activation_type": "QUInt8",
    "weight_type": "QInt8",
    "calibration_method": "MinMax",
    "operators_quantized": [
      "Conv",
      "MatMul",
      "Gemm"
    ]
  },
  "tflite_configuration": {
    "supported_ops": [
      "TFLITE_BUILTINS_INT8"
    ],
    "inference_input_type": "int8",
    "inference_output_type": "int8",
    "representative_dataset": "calibration_100.csv"
  }
}

In [17]:

class CalibrationDataReaderImpl(CalibrationDataReader):
    def __init__(self, records):
        self.records = list(records)
        self.index = 0

    def get_next(self):
        if self.index >= len(self.records):
            return None
        record = self.records[self.index]
        self.index += 1
        tensor = prepare_image_tensor(Path(record.get("absolute_path", PROJECT_ROOT / record["relative_path"])))
        return {"input_1": tensor[None, ...].astype(np.float32, copy=False)}

def build_onnx_int8_logits_model(source_path, temp_dir):
    model_proto = onnx.load(str(source_path))
    softmax_node = next((node for node in model_proto.graph.node if node.op_type == "Softmax"), None)
    if softmax_node is None:
        raise RuntimeError("ONNX baseline model does not contain a Softmax node")
    logits_name = softmax_node.input[0]
    kept_nodes = [node for node in model_proto.graph.node if node.name != softmax_node.name]
    del model_proto.graph.node[:]
    model_proto.graph.node.extend(kept_nodes)
    model_proto.graph.output.clear()
    model_proto.graph.output.extend([helper.make_tensor_value_info(logits_name, TensorProto.FLOAT, [1, 1000])])
    logits_path = temp_dir / "efficientnetb0_fp32_logits.onnx"
    onnx.save(model_proto, str(logits_path))
    return logits_path

def run_onnx_int8_benchmark(session, input_name, output_name, records, tensors):
    monitor = ResourceMonitor()
    monitor.process.cpu_percent(None)
    psutil.cpu_percent(None)
    rows = []
    latencies_ms = []
    monitor.sample(notes="onnx_int8_model_only_start")
    for record, tensor in zip(records, tensors):
        start = time.perf_counter()
        logits = session.run([output_name], {input_name: tensor[None, ...].astype(np.float32, copy=False)})[0][0]
        probabilities = softmax_np(logits)
        elapsed_ms = (time.perf_counter() - start) * 1000.0
        rows.append(probability_to_prediction_row(record, probabilities))
        latencies_ms.append(elapsed_ms)
        monitor.sample(notes="onnx_int8_model_only")
    monitor.sample(notes="onnx_int8_model_only_end")
    return pd.DataFrame(rows), summarize_latencies(latencies_ms), summarize_resource_samples(monitor.samples), latencies_ms

def run_onnx_int8_end_to_end(session, input_name, output_name, records):
    monitor = ResourceMonitor()
    monitor.process.cpu_percent(None)
    psutil.cpu_percent(None)
    rows = []
    latencies_ms = []
    monitor.sample(notes="onnx_int8_end_to_end_start")
    for record in records:
        start = time.perf_counter()
        tensor = prepare_image_tensor(Path(record.get("absolute_path", PROJECT_ROOT / record["relative_path"])))
        logits = session.run([output_name], {input_name: tensor[None, ...].astype(np.float32, copy=False)})[0][0]
        probabilities = softmax_np(logits)
        elapsed_ms = (time.perf_counter() - start) * 1000.0
        rows.append(probability_to_prediction_row(record, probabilities))
        latencies_ms.append(elapsed_ms)
        monitor.sample(notes="onnx_int8_end_to_end")
    monitor.sample(notes="onnx_int8_end_to_end_end")
    return pd.DataFrame(rows), summarize_latencies(latencies_ms), summarize_resource_samples(monitor.samples), latencies_ms

print("B?t ??u chuy?n ??i ONNX INT8")
with tempfile.TemporaryDirectory(dir=str(PROJECT_ROOT / "logs")) as tmp_dir_name:
    tmp_dir = Path(tmp_dir_name)
    logits_model_path = build_onnx_int8_logits_model(ONNX_MODEL_PATH, tmp_dir)
    quantize_static(
        model_input=str(logits_model_path),
        model_output=str(ONNX_INT8_MODEL_PATH),
        calibration_data_reader=CalibrationDataReaderImpl(calibration_records),
        quant_format=QuantFormat.QDQ,
        activation_type=QuantType.QUInt8,
        weight_type=QuantType.QInt8,
        calibrate_method=CalibrationMethod.MinMax,
        op_types_to_quantize=["Conv", "MatMul", "Gemm"],
        per_channel=True,
        reduce_range=False,
        extra_options={"ActivationSymmetric": False, "WeightSymmetric": True, "EnableSubgraph": True, "ForceQuantizeNoInputCheck": True},
    )

onnx_int8_model = onnx.load(str(ONNX_INT8_MODEL_PATH))
onnx.checker.check_model(onnx_int8_model)
onnx_int8_session = ort.InferenceSession(str(ONNX_INT8_MODEL_PATH), providers=["CPUExecutionProvider"])
onnx_int8_input_name = onnx_int8_session.get_inputs()[0].name
onnx_int8_output_name = onnx_int8_session.get_outputs()[0].name
onnx_int8_provider = onnx_int8_session.get_providers()
if onnx_int8_provider != ["CPUExecutionProvider"]:
    raise RuntimeError(f"ONNX INT8 provider must be CPUExecutionProvider, got {onnx_int8_provider}")

onnx_int8_input_shape = onnx_int8_session.get_inputs()[0].shape
onnx_int8_output_shape = onnx_int8_session.get_outputs()[0].shape
onnx_int8_validation_records = manifest_records[:10]
onnx_int8_rows = []
onnx_int8_top1 = 0
onnx_int8_top5 = 0
onnx_int8_max_abs_diffs = []
onnx_int8_mean_abs_diffs = []
for record, tensor in zip(onnx_int8_validation_records, preloaded_tensors[:10]):
    tf_probs = tf_model(tf.convert_to_tensor(tensor[None, ...], dtype=tf.float32), training=False).numpy()[0]
    logits = onnx_int8_session.run([onnx_int8_output_name], {onnx_int8_input_name: tensor[None, ...].astype(np.float32, copy=False)})[0][0]
    probs = softmax_np(logits)
    if not np.isfinite(probs).all():
        raise ValueError(f"ONNX INT8 validation produced non-finite output for {record['relative_path']}")
    max_abs_diff = float(np.max(np.abs(tf_probs - probs)))
    mean_abs_diff = float(np.mean(np.abs(tf_probs - probs)))
    onnx_int8_max_abs_diffs.append(max_abs_diff)
    onnx_int8_mean_abs_diffs.append(mean_abs_diff)
    tf_top1 = int(np.argmax(tf_probs))
    tf_top5 = set(np.argsort(tf_probs)[-5:][::-1].tolist())
    onnx_top1 = int(np.argmax(probs))
    onnx_top5 = set(np.argsort(probs)[-5:][::-1].tolist())
    onnx_int8_top1 += int(tf_top1 == onnx_top1)
    onnx_int8_top5 += int(len(tf_top5 & onnx_top5) > 0)
    onnx_int8_rows.append({"sample_id": record["sample_id"], "relative_path": record["relative_path"], "tf_top1": tf_top1, "onnx_top1": onnx_top1, "top1_agreement": bool(tf_top1 == onnx_top1), "max_abs_diff": max_abs_diff, "mean_abs_diff": mean_abs_diff})

onnx_int8_validation_df = pd.DataFrame(onnx_int8_rows)
onnx_int8_validation_top1_agreement = round(float(onnx_int8_top1 / 10), 6)
onnx_int8_validation_top5_agreement = round(float(onnx_int8_top5 / 10), 6)
onnx_int8_subset_max_abs_diff = round(float(np.max(onnx_int8_max_abs_diffs)), 12)
onnx_int8_subset_mean_abs_diff = round(float(np.mean(onnx_int8_mean_abs_diffs)), 12)
onnx_int8_validation_status = "PASS" if onnx_int8_validation_top1_agreement >= 0.97 else "DEGRADED_ACCURACY"

print("C?u h?nh ONNX INT8:")
print(json.dumps({"quantization_format": "QDQ", "activation_type": "QUInt8", "weight_type": "QInt8", "calibration_method": "MinMax", "operators_quantized": ["Conv", "MatMul", "Gemm"], "provider": onnx_int8_provider, "input_shape": onnx_int8_input_shape, "output_shape": onnx_int8_output_shape, "model_size_mb": round(ONNX_INT8_MODEL_PATH.stat().st_size / (1024 ** 2), 2)}, indent=2, ensure_ascii=False))
print("Ki?m tra t??ng ???ng s? h?c ONNX INT8 tr?n 10 ?nh:")
print(onnx_int8_validation_df.to_string(index=False))
print({"status": onnx_int8_validation_status, "subset_max_abs_diff": onnx_int8_subset_max_abs_diff, "subset_mean_abs_diff": onnx_int8_subset_mean_abs_diff, "subset_top1_agreement": onnx_int8_validation_top1_agreement, "subset_top5_agreement": onnx_int8_validation_top5_agreement})


B?t ??u chuy?n ??i ONNX INT8

D:\DAT301m\slot17\.venv-slot17\Lib\site-packages\onnxruntime\quantization\base_quantizer.py:232: RuntimeWarning: invalid value encountered in cast
  quantized_data = (np.asarray(bias_data) / bias_scale).round().astype(np.int32)


C?u h?nh ONNX INT8:

{
  "quantization_format": "QDQ",
  "activation_type": "QUInt8",
  "weight_type": "QInt8",
  "calibration_method": "MinMax",
  "operators_quantized": [
    "Conv",
    "MatMul",
    "Gemm"
  ],
  "provider": [
    "CPUExecutionProvider"
  ],
  "input_shape": [
    "unk__1292",
    224,
    224,
    3
  ],
  "output_shape": [
    1,
    1000
  ],
  "model_size_mb": 5.8
}

Ki?m tra t??ng ???ng s? h?c ONNX INT8 tr?n 10 ?nh:

sample_id                                               relative_path  tf_top1  onnx_top1  top1_agreement  max_abs_diff  mean_abs_diff
    b0000  data/raw/imagenette2-320/val/n01440764/n01440764_1310.JPEG        0          0            True      0.532185       0.001138
    b0001 data/raw/imagenette2-320/val/n01440764/n01440764_14190.JPEG      389        389            True      0.297657       0.000596
    b0002  data/raw/imagenette2-320/val/n01440764/n01440764_8622.JPEG        0          0            True      0.065971       0.000132
    b0003  data/raw/imagenette2-320/val/n01440764/n01440764_6052.JPEG        0          0            True      0.288770       0.000579
    b0004  data/raw/imagenette2-320/val/n01440764/n01440764_8250.JPEG        0          0            True      0.131024       0.000278
    b0005  data/raw/imagenette2-320/val/n01440764/n01440764_6550.JPEG        0          0            True      0.095754       0.000192
    b0006  data/raw/imagenette2-320/val/n01440764/n0144

{'status': 'PASS', 'subset_max_abs_diff': 0.547878980637, 'subset_mean_abs_diff': 0.000467412312, 'subset_top1_agreement': 1.0, 'subset_top5_agreement': 1.0}

In [18]:

onnx_int8_predictions_df, onnx_int8_model_only_latency_summary, onnx_int8_model_only_resource_summary, onnx_int8_model_only_latencies_ms = run_onnx_int8_benchmark(
    onnx_int8_session, onnx_int8_input_name, onnx_int8_output_name, manifest_records, preloaded_tensors
)
onnx_int8_end_to_end_predictions_df, onnx_int8_end_to_end_summary, onnx_int8_end_to_end_resource_summary, onnx_int8_end_to_end_latencies_ms = run_onnx_int8_end_to_end(
    onnx_int8_session, onnx_int8_input_name, onnx_int8_output_name, manifest_records
)

if len(onnx_int8_predictions_df) != BENCHMARK_SIZE or len(onnx_int8_end_to_end_predictions_df) != BENCHMARK_SIZE:
    raise ValueError("ONNX INT8 ph?i t?o ??ng 500 prediction")
if not np.isfinite(onnx_int8_predictions_df["predicted_top1_probability"].to_numpy(dtype=np.float64)).all():
    raise ValueError("ONNX INT8 prediction c? NaN/Inf")

onnx_int8_predictions_df.to_csv(ONNX_INT8_PREDICTIONS_PATH, index=False, encoding="utf-8")

onnx_int8_top1_accuracy = round(float(onnx_int8_predictions_df["top1_correct"].mean()), 6)
onnx_int8_top5_accuracy = round(float(onnx_int8_predictions_df["top5_correct"].mean()), 6)
onnx_int8_accuracy_delta = round(float(onnx_int8_top1_accuracy - baseline_top1_accuracy), 6)
onnx_int8_speedup_vs_baseline = round(float(baseline_mean_latency_ms / onnx_int8_model_only_latency_summary["mean_latency_ms"]), 6)
onnx_int8_speedup_vs_fp32_runtime = round(float(onnx_fp32_mean_latency_ms / onnx_int8_model_only_latency_summary["mean_latency_ms"]), 6)
onnx_int8_size_reduction_pct = round(float((baseline_model_size_mb - (ONNX_INT8_MODEL_PATH.stat().st_size / (1024 ** 2))) / baseline_model_size_mb * 100.0), 6)
onnx_int8_size_reduction_vs_fp32_runtime = round(float((onnx_fp32_model_size_mb - (ONNX_INT8_MODEL_PATH.stat().st_size / (1024 ** 2))) / onnx_fp32_model_size_mb * 100.0), 6)
onnx_int8_row = {column: None for column in RESULT_COLUMNS}
onnx_int8_row.update({
    "run_id": "milestone_5_onnx_int8",
    "model_id": ONNX_INT8_MODEL_ID,
    "runtime": "ONNX Runtime",
    "precision": "INT8",
    "device": "CPU",
    "provider": "CPUExecutionProvider",
    "batch_size": BATCH_SIZE,
    "thread_count": THREAD_COUNT,
    "num_images": BENCHMARK_SIZE,
    "warmup_runs": WARMUP_RUNS,
    "model_size_mb": round(ONNX_INT8_MODEL_PATH.stat().st_size / (1024 ** 2), 2),
    "load_time_s": None,
    "total_model_only_time_s": round(float(sum(onnx_int8_model_only_latencies_ms) / 1000.0), 6),
    "total_end_to_end_time_s": round(float(sum(onnx_int8_end_to_end_latencies_ms) / 1000.0), 6),
    "mean_latency_ms": onnx_int8_model_only_latency_summary["mean_latency_ms"],
    "median_latency_ms": onnx_int8_model_only_latency_summary["median_latency_ms"],
    "p95_latency_ms": onnx_int8_model_only_latency_summary["p95_latency_ms"],
    "fps_model_only": onnx_int8_model_only_latency_summary["fps"],
    "fps_end_to_end": onnx_int8_end_to_end_summary["fps"],
    "top1_accuracy": onnx_int8_top1_accuracy,
    "top5_accuracy": onnx_int8_top5_accuracy,
    "top1_agreement": onnx_int8_validation_top1_agreement,
    "max_abs_output_diff": onnx_int8_subset_max_abs_diff,
    "mean_abs_output_diff": onnx_int8_subset_mean_abs_diff,
    "ram_avg_mb": onnx_int8_model_only_resource_summary["ram_avg_mb"],
    "ram_peak_mb": onnx_int8_model_only_resource_summary["ram_peak_mb"],
    "cpu_process_avg_pct": onnx_int8_model_only_resource_summary["cpu_process_avg_pct"],
    "cpu_process_peak_pct": onnx_int8_model_only_resource_summary["cpu_process_peak_pct"],
    "cpu_system_avg_pct": onnx_int8_model_only_resource_summary["cpu_system_avg_pct"],
    "cpu_system_peak_pct": onnx_int8_model_only_resource_summary["cpu_system_peak_pct"],
    "gpu_avg_pct": onnx_int8_model_only_resource_summary["gpu_avg_pct"],
    "gpu_peak_pct": onnx_int8_model_only_resource_summary["gpu_peak_pct"],
    "vram_avg_mb": onnx_int8_model_only_resource_summary["vram_avg_mb"],
    "vram_peak_mb": onnx_int8_model_only_resource_summary["vram_peak_mb"],
    "speedup_vs_baseline": onnx_int8_speedup_vs_baseline,
    "size_reduction_pct": onnx_int8_size_reduction_pct,
    "accuracy_delta": onnx_int8_accuracy_delta,
    "notes": json.dumps({"quantization_format": "QDQ", "activation_type": "QUInt8", "weight_type": "QInt8", "calibration_method": "MinMax", "operators_quantized": ["Conv", "MatMul", "Gemm"], "speedup_vs_fp32_runtime": onnx_int8_speedup_vs_fp32_runtime, "size_reduction_vs_fp32_runtime": onnx_int8_size_reduction_vs_fp32_runtime, "validation_status": onnx_int8_validation_status}, ensure_ascii=False),
})
print("M?u prediction ONNX INT8:")
print(onnx_int8_predictions_df.head(5).to_string(index=False))
print("B?ng so s?nh TF FP32 vs ONNX FP32 vs ONNX INT8:")
print(pd.DataFrame([baseline_row, onnx_fp32_row, onnx_int8_row])[["model_id", "runtime", "precision", "provider", "model_size_mb", "mean_latency_ms", "fps_model_only", "top1_accuracy", "top1_agreement", "size_reduction_pct", "speedup_vs_baseline", "accuracy_delta"]].to_string(index=False))
print(json.dumps({"prediction_file": str(ONNX_INT8_PREDICTIONS_PATH), "status": onnx_int8_validation_status}, indent=2, ensure_ascii=False))


M?u prediction ONNX INT8:

sample_id                                               relative_path  ground_truth_index  predicted_top1_index predicted_top1_class  predicted_top1_probability            top5_indices                                                   top5_classes                                                                                             top5_probabilities  top1_correct  top5_correct
    b0000  data/raw/imagenette2-320/val/n01440764/n01440764_1310.JPEG                   0                     0                tench                    0.172637  [0, 397, 392, 394, 29]      ["tench", "puffer", "rock_beauty", "sturgeon", "axolotl"]        [0.172637477517128, 0.07700133323669434, 0.04864666610956192, 0.03656970337033272, 0.02673482894897461]          True          True
    b0001 data/raw/imagenette2-320/val/n01440764/n01440764_14190.JPEG                   0                   389           barracouta                    0.444945 [389, 395, 394, 390, 3]        ["barracouta", "gar", "sturgeon", 

B?ng so s?nh TF FP32 vs ONNX FP32 vs ONNX INT8:

                    model_id          runtime precision             provider  model_size_mb  mean_latency_ms  fps_model_only  top1_accuracy  top1_agreement  size_reduction_pct  speedup_vs_baseline  accuracy_delta
efficientnetb0_fp32_baseline TensorFlow/Keras      FP32     TensorFlow/Keras          21.04       140.001183        7.142797          0.836             1.0            0.000000             1.000000           0.000
    efficientnetb0_fp32_onnx     ONNX Runtime      FP32 CPUExecutionProvider          20.17         9.016135      110.912265          0.836             1.0            4.134981            15.527848           0.000
    efficientnetb0_int8_onnx     ONNX Runtime      INT8 CPUExecutionProvider           5.80        11.630454       85.981165          0.702             1.0           72.434035            12.037465          -0.134

{
  "prediction_file": "D:\\DAT301m\\slot17\\results\\predictions_onnx_int8.csv",
  "status": "PASS"
}

In [19]:

def representative_dataset():
    for record in calibration_records:
        tensor = prepare_image_tensor(Path(record.get("absolute_path", PROJECT_ROOT / record["relative_path"])))
        yield [tensor[None, ...].astype(np.float32, copy=False)]

print("B?t ??u chuy?n ??i TFLite INT8")
tflite_converter = tf.lite.TFLiteConverter.from_keras_model(tf_model)
tflite_converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_converter.representative_dataset = representative_dataset
tflite_converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
tflite_converter.inference_input_type = tf.int8
tflite_converter.inference_output_type = tf.int8
tflite_int8_model_bytes = tflite_converter.convert()
TFLITE_INT8_MODEL_PATH.write_bytes(tflite_int8_model_bytes)
tflite_int8_model_size_mb = round(TFLITE_INT8_MODEL_PATH.stat().st_size / (1024 ** 2), 2)

tflite_int8_interpreter = tf.lite.Interpreter(model_path=str(TFLITE_INT8_MODEL_PATH), num_threads=THREAD_COUNT)
tflite_int8_interpreter.allocate_tensors()
tflite_int8_input_details_raw = tflite_int8_interpreter.get_input_details()[0]
tflite_int8_output_details_raw = tflite_int8_interpreter.get_output_details()[0]
tflite_int8_input_details = {"name": tflite_int8_input_details_raw["name"], "shape": tflite_int8_input_details_raw["shape"].tolist(), "dtype": str(tflite_int8_input_details_raw["dtype"].__name__), "index": int(tflite_int8_input_details_raw["index"]), "scale": float(tflite_int8_input_details_raw["quantization"][0]), "zero_point": int(tflite_int8_input_details_raw["quantization"][1])}
tflite_int8_output_details = {"name": tflite_int8_output_details_raw["name"], "shape": tflite_int8_output_details_raw["shape"].tolist(), "dtype": str(tflite_int8_output_details_raw["dtype"].__name__), "index": int(tflite_int8_output_details_raw["index"]), "scale": float(tflite_int8_output_details_raw["quantization"][0]), "zero_point": int(tflite_int8_output_details_raw["quantization"][1])}
if tflite_int8_input_details["dtype"] != "int8" or tflite_int8_output_details["dtype"] != "int8":
    raise RuntimeError(f"TFLite INT8 dtype kh?ng ??ng: {tflite_int8_input_details['dtype']} -> {tflite_int8_output_details['dtype']}")
if tflite_int8_input_details["shape"] != [1, 224, 224, 3] or tflite_int8_output_details["shape"] != [1, 1000]:
    raise RuntimeError(f"TFLite INT8 shape kh?ng ??ng: {tflite_int8_input_details['shape']} -> {tflite_int8_output_details['shape']}")

tflite_int8_validation_rows = []
tflite_int8_top1 = 0
tflite_int8_top5 = 0
tflite_int8_max_abs_diffs = []
tflite_int8_mean_abs_diffs = []
for record, tensor in zip(manifest_records[:10], preloaded_tensors[:10]):
    tf_probs = tf_model(tf.convert_to_tensor(tensor[None, ...], dtype=tf.float32), training=False).numpy()[0]
    q_tensor = quantize_tflite_tensor(tensor, tflite_int8_input_details)
    tflite_int8_interpreter.set_tensor(tflite_int8_input_details["index"], q_tensor[None, ...])
    tflite_int8_interpreter.invoke()
    q_output = tflite_int8_interpreter.get_tensor(tflite_int8_output_details["index"])[0]
    probs = dequantize_tflite_tensor(q_output, tflite_int8_output_details)
    max_abs_diff = float(np.max(np.abs(tf_probs - probs)))
    mean_abs_diff = float(np.mean(np.abs(tf_probs - probs)))
    tflite_int8_max_abs_diffs.append(max_abs_diff)
    tflite_int8_mean_abs_diffs.append(mean_abs_diff)
    tf_top1 = int(np.argmax(tf_probs))
    tf_top5 = set(np.argsort(tf_probs)[-5:][::-1].tolist())
    tflite_top1 = int(np.argmax(probs))
    tflite_top5 = set(np.argsort(probs)[-5:][::-1].tolist())
    tflite_int8_top1 += int(tf_top1 == tflite_top1)
    tflite_int8_top5 += int(len(tf_top5 & tflite_top5) > 0)
    tflite_int8_validation_rows.append({"sample_id": record["sample_id"], "relative_path": record["relative_path"], "tf_top1": tf_top1, "tflite_top1": tflite_top1, "top1_agreement": bool(tf_top1 == tflite_top1), "max_abs_diff": max_abs_diff, "mean_abs_diff": mean_abs_diff})

tflite_int8_validation_df = pd.DataFrame(tflite_int8_validation_rows)
tflite_int8_validation_top1_agreement = round(float(tflite_int8_top1 / 10), 6)
tflite_int8_validation_top5_agreement = round(float(tflite_int8_top5 / 10), 6)
tflite_int8_subset_max_abs_diff = round(float(np.max(tflite_int8_max_abs_diffs)), 12)
tflite_int8_subset_mean_abs_diff = round(float(np.mean(tflite_int8_mean_abs_diffs)), 12)
tflite_int8_validation_status = "PASS" if tflite_int8_validation_top1_agreement >= 0.97 else "DEGRADED_ACCURACY"

print("C?u h?nh TFLite INT8:")
print(json.dumps({"input": tflite_int8_input_details, "output": tflite_int8_output_details, "model_size_mb": tflite_int8_model_size_mb, "representative_dataset_rows": len(calibration_records), "supported_ops": ["TFLITE_BUILTINS_INT8"], "interpreter_threads": THREAD_COUNT}, indent=2, ensure_ascii=False))
print("Ki?m tra t??ng ???ng s? h?c TFLite INT8 tr?n 10 ?nh:")
print(tflite_int8_validation_df.to_string(index=False))
print({"status": tflite_int8_validation_status, "subset_max_abs_diff": tflite_int8_subset_max_abs_diff, "subset_mean_abs_diff": tflite_int8_subset_mean_abs_diff, "subset_top1_agreement": tflite_int8_validation_top1_agreement, "subset_top5_agreement": tflite_int8_validation_top5_agreement})


B?t ??u chuy?n ??i TFLite INT8

INFO:tensorflow:Assets written to: C:\Users\klein\AppData\Local\Temp\tmpn6b37p92\assets


INFO:tensorflow:Assets written to: C:\Users\klein\AppData\Local\Temp\tmpn6b37p92\assets


D:\DAT301m\slot17\.venv-slot17\Lib\site-packages\tensorflow\lite\python\convert.py:953: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


C?u h?nh TFLite INT8:

{
  "input": {
    "name": "serving_default_input_1:0",
    "shape": [
      1,
      224,
      224,
      3
    ],
    "dtype": "int8",
    "index": 0,
    "scale": 1.0,
    "zero_point": -128
  },
  "output": {
    "name": "StatefulPartitionedCall:0",
    "shape": [
      1,
      1000
    ],
    "dtype": "int8",
    "index": 493,
    "scale": 0.00390625,
    "zero_point": -128
  },
  "model_size_mb": 5.9,
  "representative_dataset_rows": 100,
  "supported_ops": [
    "TFLITE_BUILTINS_INT8"
  ],
  "interpreter_threads": 1
}

Ki?m tra t??ng ???ng s? h?c TFLite INT8 tr?n 10 ?nh:

sample_id                                               relative_path  tf_top1  tflite_top1  top1_agreement  max_abs_diff  mean_abs_diff
    b0000  data/raw/imagenette2-320/val/n01440764/n01440764_1310.JPEG        0            0            True      0.501698       0.000921
    b0001 data/raw/imagenette2-320/val/n01440764/n01440764_14190.JPEG      389          389            True      0.234790       0.000458
    b0002  data/raw/imagenette2-320/val/n01440764/n01440764_8622.JPEG        0            0            True      0.017585       0.000043
    b0003  data/raw/imagenette2-320/val/n01440764/n01440764_6052.JPEG        0            0            True      0.230569       0.000427
    b0004  data/raw/imagenette2-320/val/n01440764/n01440764_8250.JPEG        0            0            True      0.025096       0.000101
    b0005  data/raw/imagenette2-320/val/n01440764/n01440764_6550.JPEG        0            0            True      0.075012       0.000116
    b0006  data/raw/imagenette2-320/val/n

{'status': 'PASS', 'subset_max_abs_diff': 0.501697719097, 'subset_mean_abs_diff': 0.000288663284, 'subset_top1_agreement': 1.0, 'subset_top5_agreement': 1.0}

In [20]:

def run_tflite_int8_benchmark(interpreter, input_details, output_details, records, tensors):
    monitor = ResourceMonitor()
    monitor.process.cpu_percent(None)
    psutil.cpu_percent(None)
    rows = []
    latencies_ms = []
    monitor.sample(notes="tflite_int8_model_only_start")
    for record, tensor in zip(records, tensors):
        start = time.perf_counter()
        q_tensor = quantize_tflite_tensor(tensor, input_details)
        interpreter.set_tensor(input_details["index"], q_tensor[None, ...])
        interpreter.invoke()
        q_output = interpreter.get_tensor(output_details["index"])[0]
        probabilities = dequantize_tflite_tensor(q_output, output_details)
        elapsed_ms = (time.perf_counter() - start) * 1000.0
        rows.append(probability_to_prediction_row(record, probabilities))
        latencies_ms.append(elapsed_ms)
        monitor.sample(notes="tflite_int8_model_only")
    monitor.sample(notes="tflite_int8_model_only_end")
    return pd.DataFrame(rows), summarize_latencies(latencies_ms), summarize_resource_samples(monitor.samples), latencies_ms

def run_tflite_int8_end_to_end(interpreter, input_details, output_details, records):
    monitor = ResourceMonitor()
    monitor.process.cpu_percent(None)
    psutil.cpu_percent(None)
    rows = []
    latencies_ms = []
    monitor.sample(notes="tflite_int8_end_to_end_start")
    for record in records:
        start = time.perf_counter()
        tensor = prepare_image_tensor(Path(record.get("absolute_path", PROJECT_ROOT / record["relative_path"])))
        q_tensor = quantize_tflite_tensor(tensor, input_details)
        interpreter.set_tensor(input_details["index"], q_tensor[None, ...])
        interpreter.invoke()
        q_output = interpreter.get_tensor(output_details["index"])[0]
        probabilities = dequantize_tflite_tensor(q_output, output_details)
        elapsed_ms = (time.perf_counter() - start) * 1000.0
        rows.append(probability_to_prediction_row(record, probabilities))
        latencies_ms.append(elapsed_ms)
        monitor.sample(notes="tflite_int8_end_to_end")
    monitor.sample(notes="tflite_int8_end_to_end_end")
    return pd.DataFrame(rows), summarize_latencies(latencies_ms), summarize_resource_samples(monitor.samples), latencies_ms

tflite_int8_predictions_df, tflite_int8_model_only_latency_summary, tflite_int8_model_only_resource_summary, tflite_int8_model_only_latencies_ms = run_tflite_int8_benchmark(
    tflite_int8_interpreter, tflite_int8_input_details, tflite_int8_output_details, manifest_records, preloaded_tensors
)
tflite_int8_end_to_end_predictions_df, tflite_int8_end_to_end_summary, tflite_int8_end_to_end_resource_summary, tflite_int8_end_to_end_latencies_ms = run_tflite_int8_end_to_end(
    tflite_int8_interpreter, tflite_int8_input_details, tflite_int8_output_details, manifest_records
)

if len(tflite_int8_predictions_df) != BENCHMARK_SIZE or len(tflite_int8_end_to_end_predictions_df) != BENCHMARK_SIZE:
    raise ValueError("TFLite INT8 ph?i t?o ??ng 500 prediction")
if not np.isfinite(tflite_int8_predictions_df["predicted_top1_probability"].to_numpy(dtype=np.float64)).all():
    raise ValueError("TFLite INT8 prediction c? NaN/Inf")

tflite_int8_predictions_df.to_csv(TFLITE_INT8_PREDICTIONS_PATH, index=False, encoding="utf-8")

tflite_int8_top1_accuracy = round(float(tflite_int8_predictions_df["top1_correct"].mean()), 6)
tflite_int8_top5_accuracy = round(float(tflite_int8_predictions_df["top5_correct"].mean()), 6)
tflite_int8_accuracy_delta = round(float(tflite_int8_top1_accuracy - baseline_top1_accuracy), 6)
tflite_int8_speedup_vs_baseline = round(float(baseline_mean_latency_ms / tflite_int8_model_only_latency_summary["mean_latency_ms"]), 6)
tflite_int8_speedup_vs_fp32_runtime = round(float(tflite_fp32_mean_latency_ms / tflite_int8_model_only_latency_summary["mean_latency_ms"]), 6)
tflite_int8_size_reduction_pct = round(float((baseline_model_size_mb - tflite_int8_model_size_mb) / baseline_model_size_mb * 100.0), 6)
tflite_int8_size_reduction_vs_fp32_runtime = round(float((tflite_fp32_model_size_mb - tflite_int8_model_size_mb) / tflite_fp32_model_size_mb * 100.0), 6)
tflite_int8_vs_onnx_fp32_ratio = round(float(onnx_fp32_mean_latency_ms / tflite_int8_model_only_latency_summary["mean_latency_ms"]), 6)
tflite_int8_row = {column: None for column in RESULT_COLUMNS}
tflite_int8_row.update({
    "run_id": "milestone_5_tflite_int8",
    "model_id": TFLITE_INT8_MODEL_ID,
    "runtime": "TFLite Interpreter",
    "precision": "INT8",
    "device": "CPU",
    "provider": "TFLite Interpreter",
    "batch_size": BATCH_SIZE,
    "thread_count": THREAD_COUNT,
    "num_images": BENCHMARK_SIZE,
    "warmup_runs": WARMUP_RUNS,
    "model_size_mb": tflite_int8_model_size_mb,
    "load_time_s": None,
    "total_model_only_time_s": round(float(sum(tflite_int8_model_only_latencies_ms) / 1000.0), 6),
    "total_end_to_end_time_s": round(float(sum(tflite_int8_end_to_end_latencies_ms) / 1000.0), 6),
    "mean_latency_ms": tflite_int8_model_only_latency_summary["mean_latency_ms"],
    "median_latency_ms": tflite_int8_model_only_latency_summary["median_latency_ms"],
    "p95_latency_ms": tflite_int8_model_only_latency_summary["p95_latency_ms"],
    "fps_model_only": tflite_int8_model_only_latency_summary["fps"],
    "fps_end_to_end": tflite_int8_end_to_end_summary["fps"],
    "top1_accuracy": tflite_int8_top1_accuracy,
    "top5_accuracy": tflite_int8_top5_accuracy,
    "top1_agreement": tflite_int8_validation_top1_agreement,
    "max_abs_output_diff": tflite_int8_subset_max_abs_diff,
    "mean_abs_output_diff": tflite_int8_subset_mean_abs_diff,
    "ram_avg_mb": tflite_int8_model_only_resource_summary["ram_avg_mb"],
    "ram_peak_mb": tflite_int8_model_only_resource_summary["ram_peak_mb"],
    "cpu_process_avg_pct": tflite_int8_model_only_resource_summary["cpu_process_avg_pct"],
    "cpu_process_peak_pct": tflite_int8_model_only_resource_summary["cpu_process_peak_pct"],
    "cpu_system_avg_pct": tflite_int8_model_only_resource_summary["cpu_system_avg_pct"],
    "cpu_system_peak_pct": tflite_int8_model_only_resource_summary["cpu_system_peak_pct"],
    "gpu_avg_pct": tflite_int8_model_only_resource_summary["gpu_avg_pct"],
    "gpu_peak_pct": tflite_int8_model_only_resource_summary["gpu_peak_pct"],
    "vram_avg_mb": tflite_int8_model_only_resource_summary["vram_avg_mb"],
    "vram_peak_mb": tflite_int8_model_only_resource_summary["vram_peak_mb"],
    "speedup_vs_baseline": tflite_int8_speedup_vs_baseline,
    "size_reduction_pct": tflite_int8_size_reduction_pct,
    "accuracy_delta": tflite_int8_accuracy_delta,
    "notes": json.dumps({"quantization_mode": "full_integer", "input_dtype": tflite_int8_input_details["dtype"], "output_dtype": tflite_int8_output_details["dtype"], "input_scale": tflite_int8_input_details["scale"], "input_zero_point": tflite_int8_input_details["zero_point"], "output_scale": tflite_int8_output_details["scale"], "output_zero_point": tflite_int8_output_details["zero_point"], "speedup_vs_fp32_runtime": tflite_int8_speedup_vs_fp32_runtime, "speedup_vs_onnx_fp32": tflite_int8_vs_onnx_fp32_ratio, "size_reduction_vs_fp32_runtime": tflite_int8_size_reduction_vs_fp32_runtime, "validation_status": tflite_int8_validation_status}, ensure_ascii=False),
})
print("M?u prediction TFLite INT8:")
print(tflite_int8_predictions_df.head(5).to_string(index=False))
print("B?ng so s?nh TF FP32 vs ONNX FP32 vs TFLite FP32 vs ONNX INT8 vs TFLite INT8:")
print(pd.DataFrame([baseline_row, onnx_fp32_row, tflite_fp32_row, onnx_int8_row, tflite_int8_row])[["model_id", "runtime", "precision", "provider", "model_size_mb", "mean_latency_ms", "fps_model_only", "top1_accuracy", "top1_agreement", "size_reduction_pct", "speedup_vs_baseline", "accuracy_delta"]].to_string(index=False))
print(json.dumps({"prediction_file": str(TFLITE_INT8_PREDICTIONS_PATH), "input_details": tflite_int8_input_details, "output_details": tflite_int8_output_details, "status": tflite_int8_validation_status}, indent=2, ensure_ascii=False))

milestone5_accuracy_status = "COMPLETED_WITH_ACCURACY_DEGRADATION" if any([onnx_int8_validation_status != "PASS", tflite_int8_validation_status != "PASS", onnx_int8_accuracy_delta < -0.02, tflite_int8_accuracy_delta < -0.02, onnx_int8_validation_top1_agreement < 0.97, tflite_int8_validation_top1_agreement < 0.97]) else "READY_FOR_MILESTONE_6"
updated_benchmark_df = update_benchmark_results([onnx_int8_row, tflite_int8_row])
if len(updated_benchmark_df) != 5:
    raise ValueError(f"benchmark_results.csv ph?i c? ??ng 5 row, hi?n c? {len(updated_benchmark_df)}")
print("B?ng benchmark cu?i c?ng:")
print(updated_benchmark_df[["model_id", "runtime", "precision", "provider", "model_size_mb", "mean_latency_ms", "fps_model_only", "top1_accuracy", "top1_agreement", "size_reduction_pct", "speedup_vs_baseline", "accuracy_delta"]].to_string(index=False))
print(json.dumps({"status": milestone5_accuracy_status, "onnx_int8_path": str(ONNX_INT8_MODEL_PATH), "onnx_int8_size_mb": round(ONNX_INT8_MODEL_PATH.stat().st_size / (1024 ** 2), 2), "tflite_int8_path": str(TFLITE_INT8_MODEL_PATH), "tflite_int8_size_mb": tflite_int8_model_size_mb, "tflite_int8_dtype": tflite_int8_input_details, "prediction_files": {"onnx_int8": str(ONNX_INT8_PREDICTIONS_PATH), "tflite_int8": str(TFLITE_INT8_PREDICTIONS_PATH)}, "benchmark_file": str(BASELINE_BENCHMARK_PATH), "notebook_output_status": "SAVED_INPLACE"}, indent=2, ensure_ascii=False, default=str))


M?u prediction TFLite INT8:

sample_id                                               relative_path  ground_truth_index  predicted_top1_index predicted_top1_class  predicted_top1_probability            top5_indices                                                      top5_classes                                        top5_probabilities  top1_correct  top5_correct
    b0000  data/raw/imagenette2-320/val/n01440764/n01440764_1310.JPEG                   0                     0                tench                    0.203125  [0, 397, 394, 29, 395]                 ["tench", "puffer", "sturgeon", "axolotl", "gar"]       [0.203125, 0.09375, 0.0546875, 0.03515625, 0.03125]          True          True
    b0001 data/raw/imagenette2-320/val/n01440764/n01440764_14190.JPEG                   0                   389           barracouta                    0.507812 [389, 395, 394, 390, 0]                 ["barracouta", "gar", "sturgeon", "eel", "tench"] [0.5078125, 0.16015625, 0.1484375, 0.01171875, 0.0078125]         False    

B?ng so s?nh TF FP32 vs ONNX FP32 vs TFLite FP32 vs ONNX INT8 vs TFLite INT8:

                    model_id            runtime precision             provider  model_size_mb  mean_latency_ms  fps_model_only  top1_accuracy  top1_agreement  size_reduction_pct  speedup_vs_baseline  accuracy_delta
efficientnetb0_fp32_baseline   TensorFlow/Keras      FP32     TensorFlow/Keras          21.04       140.001183        7.142797          0.836             1.0            0.000000             1.000000           0.000
    efficientnetb0_fp32_onnx       ONNX Runtime      FP32 CPUExecutionProvider          20.17         9.016135      110.912265          0.836             1.0            4.134981            15.527848           0.000
  efficientnetb0_fp32_tflite TFLite Interpreter      FP32               TFLite          20.18        54.782744       18.253923          0.836             1.0            4.087452             2.555571           0.000
    efficientnetb0_int8_onnx       ONNX Runtime      INT8 CPUExecutionProvider           5.80        11.630454       85.981165          0.70

{
  "prediction_file": "D:\\DAT301m\\slot17\\results\\predictions_tflite_int8.csv",
  "input_details": {
    "name": "serving_default_input_1:0",
    "shape": [
      1,
      224,
      224,
      3
    ],
    "dtype": "int8",
    "index": 0,
    "scale": 1.0,
    "zero_point": -128
  },
  "output_details": {
    "name": "StatefulPartitionedCall:0",
    "shape": [
      1,
      1000
    ],
    "dtype": "int8",
    "index": 493,
    "scale": 0.00390625,
    "zero_point": -128
  },
  "status": "PASS"
}

C:\Users\klein\AppData\Local\Temp\ipykernel_17156\4151950178.py:77: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  updated_df = pd.concat([existing_df, pd.DataFrame(new_rows)], ignore_index=True)


B?ng benchmark cu?i c?ng:

                    model_id            runtime precision             provider  model_size_mb  mean_latency_ms  fps_model_only  top1_accuracy  top1_agreement  size_reduction_pct  speedup_vs_baseline  accuracy_delta
efficientnetb0_fp32_baseline   TensorFlow/Keras      FP32     TensorFlow/Keras          21.04       140.001183        7.142797          0.836             1.0            0.000000             1.000000           0.000
    efficientnetb0_fp32_onnx       ONNX Runtime      FP32 CPUExecutionProvider          20.17         9.016135      110.912265          0.836             1.0            4.134981            15.527848           0.000
  efficientnetb0_fp32_tflite TFLite Interpreter      FP32               TFLite          20.18        54.782744       18.253923          0.836             1.0            4.087452             2.555571           0.000
    efficientnetb0_int8_onnx       ONNX Runtime      INT8 CPUExecutionProvider           5.80        11.630454       85.981165          0.70

{
  "status": "COMPLETED_WITH_ACCURACY_DEGRADATION",
  "onnx_int8_path": "D:\\DAT301m\\slot17\\models\\onnx\\efficientnetb0_int8.onnx",
  "onnx_int8_size_mb": 5.8,
  "tflite_int8_path": "D:\\DAT301m\\slot17\\models\\tflite\\efficientnetb0_int8.tflite",
  "tflite_int8_size_mb": 5.9,
  "tflite_int8_dtype": {
    "name": "serving_default_input_1:0",
    "shape": [
      1,
      224,
      224,
      3
    ],
    "dtype": "int8",
    "index": 0,
    "scale": 1.0,
    "zero_point": -128
  },
  "prediction_files": {
    "onnx_int8": "D:\\DAT301m\\slot17\\results\\predictions_onnx_int8.csv",
    "tflite_int8": "D:\\DAT301m\\slot17\\results\\predictions_tflite_int8.csv"
  },
  "benchmark_file": "D:\\DAT301m\\slot17\\results\\benchmark_results.csv",
  "notebook_output_status": "SAVED_INPLACE"
}